In [8]:
# general imports
import os
import scipy.stats as stats
from itertools import product, combinations_with_replacement
import pickle
import ast
from unidecode import unidecode
import sys
import pandas as pd
import json

# plotting imports
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import plotly.graph_objects as go
# kaleido is necessary for saving plots made with plotly
# this needs to be version 0.1 (pip install --upgrade kaleido=="0.1.0.post1") because higher versions crash
import kaleido 

# local imports
from processing_utils import convert_to_two_level_multiindex, normalize_df_to_rpm
from unicode_utils import apply_greek_labels_heatmap, introduce_greek_letters
from constants import CELL_TYPE_COLORS, ALL_CELL_TYPES
from plotting_utils import cm_to_inches, setup_plotting_environment
setup_plotting_environment()

ALL_CELL_TYPES_EXCLUDE = [cell for cell in ALL_CELL_TYPES if cell not in ["ILC", "pDC"]]

# set up folders
input_data_folder = f"../../data/"
os.makedirs(input_data_folder, exist_ok=True)

base_plot_folder = f"./plots/"
os.makedirs(base_plot_folder, exist_ok=True)

intermediate_output_folder = f"./output/"
os.makedirs(intermediate_output_folder, exist_ok=True)

output_tables_folder = f"./output_tables/"
os.makedirs(output_tables_folder, exist_ok=True)

# 1 - Load the data

In [2]:
plot_folder = os.path.join(base_plot_folder, "1_data_loading")
os.makedirs(plot_folder, exist_ok=True)

In [3]:
# info on cytokines
cytokine_info = pd.read_excel("../annotations/20250125_cytokine_info_with_functional_classification_LV.xlsx", index_col=0, sheet_name="all_cytokines")
cytokine_info["single_gene"] = cytokine_info["gene"].str.split(",").str[0]

cytokine_info_parse = cytokine_info[cytokine_info["parse_cytokine"]]
cytokine_info_order = pd.read_excel("../annotations/20250125_cytokine_info_with_functional_classification_LV.xlsx", index_col=0, sheet_name="suggested_order")

CYTOKINE_ORDER = list(cytokine_info_order["name"])
CYTOKINE_ORDER_WITH_PBS = ["PBS"] + CYTOKINE_ORDER
SINGLE_FAMILY_ORDER = cytokine_info_order["DEG_plot_family"].unique()

In [4]:
cytokine_name_to_level_1_family = cytokine_info.reset_index().set_index("name")["family_level_1"].to_dict()
cytokine_name_to_level_2_family = cytokine_info.reset_index().set_index("name")["family_level_2"].to_dict()

cytokine_level_1_order = list(dict.fromkeys(cytokine_name_to_level_1_family.values()))
cytokine_level_2_order = list(dict.fromkeys(cytokine_name_to_level_2_family.values()))

level2_to_level1 = cytokine_info.set_index("family_level_2")["family_level_1"].to_dict()

In [5]:
# donor info - not really using this for now
# donor_names = [f"Donor{i+1}" for i in range(12)]
# donor_metadata = pd.read_excel("../annotations/donor_metadata.xlsx")
# donor_metadata["Donor number"] = donor_metadata["Donor number"].str.replace(" ", "")
# donor_metadata.index = donor_metadata["Donor number"]
# donor_metadata["age"] = 2024 - donor_metadata["Birth year"]

## 1.1 - Process pseudobulks

#### Convert pseudobulk files into a usable format

In [6]:
relevant_cell_types = [
    "CD4",
    "CD8",
    "B",
    "CD56-bright_NK",
    "Treg",
    "MAIT",
    "CD14_Mono",
    "CD16_Mono",
    "CD56-dim_NK",
    "NKT",
    "cDC",
    "pDC",
    "other",
    "ILC",
    "HSPC",
]

relevant_cell_type_conversion = {
    "Treg": "Treg",
    "CD14_Mono": "CD14 Mono",
    "HSPC": "HSPC", 
    "pDC": "pDC",
    "NKT": "NKT",
    "other": "granulocyte",
    "CD16_Mono": "CD16 Mono",
    "CD56-bright_NK": "NK CD56hi",
    "CD56-dim_NK": "NK CD56low",
    "MAIT": "MAIT",
    "cDC": "cDC",
    "ILC": "ILC",
    "CD8": "CD8 T cell",
    "B": "B cell",
    "CD4": "CD4 T cell",
    "HPSC": "HPSC"
}

In [ ]:
pseudobulk_files = os.listdir(os.path.join(input_data_folder, "pseudobulks"))

cell_types = []
cytokines = []
cell_type_counts = {}
for file in pseudobulk_files:
    cell_type = "_".join(file.split("_")[1:-1])
    cell_types.append(cell_type)
    cytokines.append(file.split("_")[-1].split(".")[0])
    cell_type_counts[cell_type] = cell_type_counts.get(cell_type, 0) + 1
cell_types = list(set(cell_types))
cytokines = sorted(list(set(cytokines)))

# make sure everything is in order
for cell_type in cell_types:
    for cytokine in cytokines:
        if not cytokine in CYTOKINE_ORDER_WITH_PBS:
            print("Not in cytokine_info_parse:", cytokine)

#### This only needs to be run once - comment out afterward

In [10]:
# # read .h5ad file
# import anndata as ad
#
# all_dfs = []
# counts_by_condition = {}
# for cell_type in relevant_cell_types:
#     for cytokine in CYTOKINE_ORDER_WITH_PBS:
#         adata = ad.read_h5ad(input_data_folder + f"pseudobulks/pbulk_{cell_type}_{cytokine}.h5ad")
#         df = adata.to_df()
#         # Split the index at _ to create a multiindex
#         df.index = pd.MultiIndex.from_tuples([idx.split('_') for idx in df.index])
#         # prepend the cell_type to the second level of the index
#         df.index = df.index.set_levels([relevant_cell_type_conversion[cell_type] + "_" + x for x in df.index.levels[1]], level=1)
#         all_dfs.append(df)
        
#         # also save counts_by_condition
#         counts_by_condition[(relevant_cell_type_conversion[cell_type], cytokine)] = adata.obs["psbulk_n_cells"]
        
# sum_gene_count_df = pd.concat(all_dfs, axis=0)
# sum_gene_count_df.fillna(0, inplace=True)

# average_sum_gene_count_df = sum_gene_count_df.groupby(level=1).mean()
# average_sum_gene_count_df = convert_to_two_level_multiindex(average_sum_gene_count_df)

# sum_gene_count_df_norm = normalize_df_to_rpm(sum_gene_count_df)
# average_norm_sum_gene_count_df = sum_gene_count_df_norm.groupby(level=1).mean()
# average_norm_sum_gene_count_df = convert_to_two_level_multiindex(average_norm_sum_gene_count_df)
# average_norm_sum_gene_count_df = average_norm_sum_gene_count_df.loc[:, ALL_CELL_TYPES]

# average_sum_gene_count_df.to_csv(input_data_folder + "pbulk_average_sum_gene_count_df.csv")
# average_norm_sum_gene_count_df.to_csv(input_data_folder + "pbulk_average_norm_sum_gene_count_df.csv")

In [11]:
average_norm_sum_gene_count_df = pd.read_csv(input_data_folder + "pbulk_average_norm_sum_gene_count_df.csv", index_col=0, header=[0, 1])
average_sum_gene_count_df = pd.read_csv(input_data_folder + "pbulk_average_sum_gene_count_df.csv", index_col=0, header=[0, 1])

average_norm_sum_gene_count_df = average_norm_sum_gene_count_df[ALL_CELL_TYPES]
average_sum_gene_count_df = average_sum_gene_count_df[ALL_CELL_TYPES]
average_norm_sum_gene_count_df_log = np.log2(average_norm_sum_gene_count_df + 1)

# separate out the control data
control_norm_gene_count_df = average_norm_sum_gene_count_df.loc[:, (slice(None), "PBS")]
# drop the "PBS" from the index
control_norm_gene_count_df.columns = control_norm_gene_count_df.columns.droplevel(1)

In [ ]:
unfiltered_genes = average_norm_sum_gene_count_df.index

inclusive_filters_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    df = average_sum_gene_count_df[cell_type]
    df = df[df.max(axis=1) > 20]
    print(cell_type, len(df))
    inclusive_filters_by_cell_type[cell_type] = df.index

## 1.2 - Load DEGs

In [13]:
edger_results = pd.read_csv(os.path.join(input_data_folder, "EdgeR_DEGs_all.csv"), index_col=0)
edger_results = edger_results.rename(columns={"celltype": "cell_type"})

In [14]:
edger_results = edger_results[edger_results["cell_type"].isin(relevant_cell_type_conversion.keys())]
edger_results["cell_type"] = edger_results["cell_type"].map(relevant_cell_type_conversion)

In [15]:
def create_edger_val_df(result_df, val_type):
    """
    Creates a DataFrame from EdgeR results with a MultiIndex of (cell_type, cytokine) columns.
    
    Parameters:
    -----------
    result_df : pandas.DataFrame
        The EdgeR results DataFrame containing gene, cell_type, cytokine, and value columns.
    val_type : str
        The column name in result_df to use as values (e.g., 'adj_p_value', 'log_fc', 'logCPM').
        
    Returns:
    --------
    pandas.DataFrame
        A pivoted DataFrame with genes as index and MultiIndex columns of (cell_type, cytokine),
        containing the specified values. Missing combinations are filled with NaN.
    """
    # rename 
    df = result_df[[val_type, "gene", "cell_type", "cytokine"]]
    # reformat to columns = (cell_type, cytokine)
    df = df.pivot(index="gene",columns=["cell_type", "cytokine"], values=val_type)
    # Create tuples of (cell_type, cytokine) for all combinations
    column_tuples = [(cell_type, cytokine) 
                    for cell_type in ALL_CELL_TYPES 
                    for cytokine in CYTOKINE_ORDER]
    df = df.reindex(pd.MultiIndex.from_tuples(column_tuples), axis=1)
    return df

edger_df_padj = create_edger_val_df(edger_results, "adj_p_value")
edger_df_log2CPM = create_edger_val_df(edger_results, "logCPM")
edger_df_log2FoldChange = create_edger_val_df(edger_results, "log_fc")

In [16]:
if not os.path.exists(os.path.join(output_tables_folder, "2_edger_df_padj.csv")):
    edger_df_padj.to_csv(os.path.join(output_tables_folder, "2_edger_df_padj.csv"))
if not os.path.exists(os.path.join(output_tables_folder, "2_edger_df_log2FoldChange.csv")):
    edger_df_log2FoldChange.to_csv(os.path.join(output_tables_folder, "2_edger_df_log2FoldChange.csv"))

## 1.3 - Filter unwanted genes


#### Option to remove mitochondrial, RNA, and ribosomal genes. Should probably be done before EdgeR calculation if at all.

In [17]:
# import pybiomart as pbm

# # hg38
# dataset_human = pbm.Dataset(name='hsapiens_gene_ensembl',  host='http://www.ensembl.org')
# annot = dataset_human.query(attributes=['chromosome_name', 'transcription_start_site', 'strand', 'external_gene_name', 'transcript_biotype'])

# # remove mitochondrial genes
# annot['Chromosome/scaffold name'] = annot['Chromosome/scaffold name'].astype(str)
# annot = annot[~annot['Chromosome/scaffold name'].str.contains('MT')]

# transcript_filter = annot['Transcript type'].isin(['protein_coding'])
# annot_filtered = annot[transcript_filter]
# protein_coding_genes = annot_filtered['Gene name'].unique()
# len(protein_coding_genes)

In [18]:
# # ribosomal genes were taken from https://www.genenames.org/data/genegroup/#!/group/1054
# with open("ribosomal_genes.json", "r") as f:
#     ribosomal_gene_info = json.load(f)
    
# ribosomal_gene_info = pd.DataFrame(ribosomal_gene_info)
# ribosomal_genes = ribosomal_gene_info["approvedSymbol"].unique()

# protein_coding_genes = [gene for gene in protein_coding_genes if not gene in ribosomal_genes]
# len(protein_coding_genes)

#### Apply the filtering (disabled for now)

In [19]:
# edger_df_padj = edger_df_padj[edger_df_padj.index.isin(protein_coding_genes)]
# edger_df_log2FoldChange = edger_df_log2FoldChange[edger_df_log2FoldChange.index.isin(protein_coding_genes)]
# average_norm_sum_gene_count_df = average_norm_sum_gene_count_df[average_norm_sum_gene_count_df.index.isin(protein_coding_genes)]
# average_norm_sum_gene_count_df_log = average_norm_sum_gene_count_df_log[average_norm_sum_gene_count_df_log.index.isin(protein_coding_genes)]
# average_sum_gene_count_df = average_sum_gene_count_df[average_sum_gene_count_df.index.isin(protein_coding_genes)]

# for cell_type in ALL_CELL_TYPES:
#     inclusive_filters_by_cell_type[cell_type] = inclusive_filters_by_cell_type[cell_type][inclusive_filters_by_cell_type[cell_type].isin(protein_coding_genes)]

## 1.4 - Overview of significance and fold changes

In [ ]:
edger_df_log2FoldChange_filtered = edger_df_log2FoldChange[edger_df_padj < 0.05]

positive_values = (edger_df_log2FoldChange_filtered > 0.5).sum(axis=0)
negative_values = (edger_df_log2FoldChange_filtered < -0.5).sum(axis=0)
total_values = positive_values + negative_values

plt.figure(figsize=cm_to_inches((5, 3)))
plt.hist(edger_df_log2FoldChange_filtered.values.flatten(), bins=np.arange(-3, 3, 0.05))

# Calculate total positive and negative values
total_positive = np.sum(positive_values)
total_negative = np.sum(negative_values)

# Add text annotations for counts
plt.text(0.05, 0.95, f"log2FC < -0.5:\n{total_negative}", 
         transform=plt.gca().transAxes, 
         verticalalignment='top', 
         horizontalalignment='left',
         fontsize=6)

plt.text(0.95, 0.95, f"log2FC > 0.5:\n{total_positive}", 
         transform=plt.gca().transAxes, 
         verticalalignment='top', 
         horizontalalignment='right',
         fontsize=6)

plt.xlabel("Log2 Fold Change")
plt.ylabel("Count")
plt.title("Distribution of significant log2 fold changes")

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"1.4_edger_df_log2FoldChange_filtered_distribution.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
positive_values = positive_values[total_values > 10]
negative_values = negative_values[total_values > 10]
ratio = positive_values / (positive_values + negative_values)
ratio_mean = ratio.groupby(level=1).mean()
ratio_std = ratio.groupby(level=1).std()
ratio_mean = ratio_mean.sort_values(ascending=False)
ratio_std = ratio_std.loc[ratio_mean.index]

# Create a bar plot of the ratio of positive to total DEGs for each cytokine with error bars
plt.figure(figsize=cm_to_inches((18, 4)))
x_pos = range(len(ratio_mean))
bars = plt.bar(x_pos, ratio_mean.values, yerr=ratio_std.values, capsize=2, error_kw={'elinewidth': 1})
# Apply Greek letters to x-tick labels
x_labels = [introduce_greek_letters(lbl) for lbl in ratio_mean.index]
plt.xticks(x_pos, x_labels, rotation=90, fontsize=5.5)
plt.yticks(fontsize=5.5)
plt.ylabel(r'mean($\frac{\mathrm{DEG}(log2FC>0.5)}{\mathrm{DEG}(log2FC>0.5) + \mathrm{DEG}(log2FC<-0.5)}$)')

plt.ylim(0, 1)
plt.xlim(-1, len(ratio_mean))  # Reduce space on left and right of x-axis
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"1.4_ratio_of_positive_to_total_DEGs.{format}"), bbox_inches="tight", dpi=300)

## 1.5 - Alyssa's reference data

In [99]:
alyssa_secretion_df = pd.read_csv("immuneXpresso_alyssa/secretion_dataframe.csv")
alyssa_immunexpresso_df = pd.read_csv("immuneXpresso_alyssa/cytokine_effects_on_cells.csv")

In [ ]:
# what fraction is in cytokine_info?
len(alyssa_immunexpresso_df[alyssa_immunexpresso_df["Cytokine"].isin(cytokine_info.index)]) / len(alyssa_immunexpresso_df)

In [95]:
alyssa_secretion_df["sumpapers"] = alyssa_secretion_df["sumpapers"].fillna(0)

In [107]:
alyssa_immunexpresso_df["total_interactions"] = alyssa_immunexpresso_df["Positive"] + alyssa_immunexpresso_df["Negative"] + alyssa_immunexpresso_df["Unknown"]

In [113]:
alyssa_immunexpresso_df_binarized = alyssa_immunexpresso_df[alyssa_immunexpresso_df["total_interactions"] > 2]

In [ ]:
alyssa_immunexpresso_df_binarized["xpresso"] = True

# 2 - Summarize results

## 2.1 -  Calculate the magnitude of the response

In [20]:
plot_folder = os.path.join(base_plot_folder, "2_result_summary")
os.makedirs(plot_folder, exist_ok=True)

### 2.1.1 - Calculate Euclidean distances

In [21]:
from scipy.spatial.distance import euclidean
from scipy.stats import mstats

def compute_response_euclidean_distance(df, cell_type, cytokine, control="PBS"):
    """
    Compute the Euclidean distance between the treatment and control cytokine response for a given cell type.

    Parameters:
    - df (pd.DataFrame): A DataFrame containing gene expression data with a MultiIndex of genes,(cell_type, cytokine).
    - cell_type (str): The specific cell type for which the distance is to be computed.
    - cytokine (str): The treatment cytokine whose response is being compared.
    - control (str, optional): The control cytokine to compare against, default is "PBS".

    Returns:
    - float: The Euclidean distance between the treatment and control responses for the specified cell type and cytokine.
    """
    df_treatment = df.loc[:, (cell_type, cytokine)]
    df_control = df.loc[:, (cell_type, control)]
    
    # compute distance
    distance = euclidean(df_treatment.values.flatten(), df_control.values.flatten())
    
    return distance

def winsorize_and_normalize_distances(distances: pd.Series):
    """
    Winsorize and normalize the distances to a scale of 0 to 100.

    Parameters:
    - distances (pd.Series): A pandas Series containing the distances to be winsorized and normalized.

    Returns:
    - pd.Series: A pandas Series of the same index as the input, containing the normalized distances.
                 The distances are first winsorized to limit extreme values, then scaled to a range of 0 to 100.
    """
    # winsorize the distances (first limit: increase smallest values, second limit: decrease largest values)
    winsorized_distances = mstats.winsorize(distances, limits=[0, 0.05])
    # make it a pandas series with the same index
    winsorized_distances = pd.Series(winsorized_distances, index=distances.index)
    
    min_val, max_val = winsorized_distances.min(), winsorized_distances.max()
    normalized_distances = 100 * (winsorized_distances - min_val) / (max_val - min_val)
    
    return normalized_distances

In [22]:
all_normalized_euclidean_distances = {}
average_norm_sum_gene_count_df_log = np.log2(average_norm_sum_gene_count_df + 1)

for cell_type in ALL_CELL_TYPES:
    distances = {}
    
    # get the inclusive filter and average normalized log2 data
    all_inclusive_filter = inclusive_filters_by_cell_type[cell_type]
    average_norm_filter = average_norm_sum_gene_count_df_log.loc[all_inclusive_filter, :]
    
    for cytokine in CYTOKINE_ORDER:
        distance = compute_response_euclidean_distance(average_norm_filter, cell_type, cytokine)
        distances[cytokine] = distance
    distances_series = pd.Series(distances)
    
    all_normalized_euclidean_distances[cell_type] = winsorize_and_normalize_distances(distances_series)
    
all_normalized_euclidean_distances = pd.DataFrame(all_normalized_euclidean_distances)

In [23]:
%%capture output

for cell_type in ALL_CELL_TYPES:
    plt.figure(figsize=(3, 2.4))
    plt.hist(all_normalized_euclidean_distances[cell_type])
    plt.xlabel("Euclidean distance")
    plt.ylabel("Count")
    plt.title(cell_type)
    plt.tight_layout()
    plt.savefig(os.path.join(plot_folder, f"2.1_normalized_euclidean_distance_{cell_type}.png"), bbox_inches="tight", dpi=300)

### 2.1.2 - Calculate p-value distances

R_{\text{strength}} = \sum_{i=1}^{N} \left| \log_2 \text{FC}_i \right| \cdot -\log_{10}(p_{\text{adj},i})

In [24]:
# get the absolute value of the log2 fold change
# the filled values will drop out in the sum
edger_df_log2FoldChange_abs = edger_df_log2FoldChange.fillna(0).abs()
edger_df_padj_fillna = edger_df_padj.fillna(1).copy()

# cap the p-values at 1e-10 and log10-transform
edger_df_padj_fillna = edger_df_padj_fillna.clip(lower=1e-10)
edger_df_padj_fillna = -np.log10(edger_df_padj_fillna)

# this implements the formula above
all_logfoldchange_padj_distances = edger_df_log2FoldChange_abs * edger_df_padj_fillna
all_logfoldchange_padj_distances = all_logfoldchange_padj_distances.sum()

# make it a dataframe (cytokine x cell_type)
all_logfoldchange_padj_distances = all_logfoldchange_padj_distances.unstack(level=0)
all_logfoldchange_padj_distances = all_logfoldchange_padj_distances.loc[CYTOKINE_ORDER]

# apply normalization and winsorization
for cell_type in ALL_CELL_TYPES:
    distances_series = all_logfoldchange_padj_distances[cell_type]
    all_logfoldchange_padj_distances[cell_type] = winsorize_and_normalize_distances(distances_series)

### 2.1.3 - Merge distance measurements and calculate thresholds for a strong response

In [ ]:
plt.figure(figsize=cm_to_inches((6, 4)))
for cell_type in ALL_CELL_TYPES:
    r = np.corrcoef(all_normalized_euclidean_distances[cell_type],
                all_logfoldchange_padj_distances[cell_type])
    plt.scatter(all_normalized_euclidean_distances[cell_type],
                all_logfoldchange_padj_distances[cell_type],
                label=f"{cell_type}: r={r[0, 1]:.2f}",
                color=CELL_TYPE_COLORS[cell_type],
                s = 2.5,
                alpha=1)
    
plt.xlabel("response magnitude\n(Euclidean)")
plt.ylabel("response magnitude\n(padj-log2FC)")
plt.legend(loc=[1,0], fontsize=6, ncol=2)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_normalized_distance_comparison.{format}"), bbox_inches="tight", dpi=300)

In [26]:
# merge the two distance measurements
all_normalized_distances_mean = (all_normalized_euclidean_distances + all_logfoldchange_padj_distances) / 200
# reset to the correct order
all_normalized_distances_mean = all_normalized_distances_mean[ALL_CELL_TYPES]
# calculate relative distances (divide by the max distance for each cytokine)
all_relative_distances_mean = all_normalized_distances_mean.div(all_normalized_distances_mean.max(axis=1), axis=0)

In [27]:
def convert_to_percentile_along_axis(df, axis, granularity=5):
    """Convert values to percentile ranks along specified axis with given granularity.
    
    Args:
        df: pandas DataFrame to convert
        axis: 0 for columns, 1 for rows 
        granularity: Size of percentile bins (e.g. 5 means bins of 0-5%, 5-10%, etc)
        
    Returns:
        DataFrame with values converted to percentile ranks
    """
    # Calculate percentile rank along axis
    percentile_df = df.rank(axis=axis, pct=True) * 100
    
    # Round to nearest granularity value
    percentile_df = (percentile_df / granularity).round() * granularity
    
    return percentile_df

all_normalized_distances_percentile = convert_to_percentile_along_axis(all_normalized_distances_mean, axis=0, granularity=5)

In [28]:
# calculate the magntiude threshold for each cell type
magnitude_threshold_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    # get the mean of the low percentile cytokines
    low_percentile = all_normalized_distances_percentile[cell_type] < 35
    low_percentile_cytokines = low_percentile.index[low_percentile]
    low_percentile_mean = all_normalized_distances_mean[cell_type][low_percentile_cytokines].mean()
    
    # set the threshold to 3x the mean of the low percentile cytokines
    magnitude_threshold_by_cell_type[cell_type] = 3*low_percentile_mean
    
strong_impact_by_cell_type = {}
for cell_type in ALL_CELL_TYPES:
    strong_impact_by_cell_type[cell_type] = all_normalized_distances_mean[cell_type] > magnitude_threshold_by_cell_type[cell_type]
strong_impact_by_cell_type = pd.DataFrame(strong_impact_by_cell_type, index=all_normalized_distances_mean.index)

In [ ]:
# Create a figure with subplots arranged in a grid
n_cell_types = len(ALL_CELL_TYPES)
n_rows = 3 
n_cols = 5
fig, axes = plt.subplots(n_rows, n_cols, figsize=cm_to_inches((17, 8)))

# Flatten axes array for easier iteration
axes_flat = axes.flatten()

for idx, cell_type in enumerate(ALL_CELL_TYPES):
    ax = axes_flat[idx]
    ax.hist(all_normalized_distances_mean[cell_type].values, bins=np.arange(0, 1.05, 0.05))
    ax.axvline(magnitude_threshold_by_cell_type[cell_type], color="red", linestyle="--", label="impact threshold" if idx == 0 else "")
    if idx % n_cols == 0:  # Only show y-label on leftmost plots
        ax.set_ylabel("cytokine count")
    else:
        ax.set_yticks([])
    if idx >= n_cols * (n_rows-1):  # Only show x-label on bottom plots
        ax.set_xlabel("response magnitude")
    else:
        ax.set_xticks([])
    ax.set_title(cell_type, fontsize=7)
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 30)
    if idx == 0:
        ax.legend(fontsize=6)

# Remove any empty subplots
for idx in range(len(ALL_CELL_TYPES), len(axes_flat)):
    fig.delaxes(axes_flat[idx])

plt.tight_layout()

# Save the figure
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_distance_distributions.{format}"), bbox_inches="tight", dpi=300)

In [30]:
all_normalized_distances_mean.to_csv(os.path.join(output_tables_folder, "2_response_magnitudes.csv"))
strong_impact_by_cell_type.to_csv(os.path.join(output_tables_folder, "2_strong_impact.csv"))

## 2.2 - Create an overview plot for DEGs and the response magnitude

In [58]:
significant_results = pd.DataFrame(index=ALL_CELL_TYPES, columns=CYTOKINE_ORDER, data=0)
for cell_line in ALL_CELL_TYPES:
    padj_df = edger_df_padj[cell_line]
    num_significant = (padj_df < 0.05).sum(axis=0)
    significant_results.loc[cell_line, CYTOKINE_ORDER] = num_significant[CYTOKINE_ORDER].values

#### Create a long df for scatter plots

In [59]:
data_long = significant_results.reset_index().rename(columns={'index': 'Cell_Type'})
data_long = data_long.melt(id_vars=['Cell_Type'], var_name='Cytokine', value_name='DEGs')

# Convert Cytokine column to a categorical type with specified order
data_long['Cytokine'] = pd.Categorical(data_long['Cytokine'], categories=CYTOKINE_ORDER, ordered=True)

# Sort data_long by the categorical order to ensure plotting follows this order
data_long = data_long.sort_values(by='Cytokine', ascending=False)

In [60]:
bin_sizes = [10, 33, 100, 333, 1000, 3333, 10000, 10E6]
bin_text = ["<10", "<33", "<100", "<333", "<1000", "<3333", "<10000", ">10000"]

def bin_DEG_numbers(x):
    if x <= bin_sizes[0]:
        return 1
    if x <= bin_sizes[1]:
        return 2
    elif x <= bin_sizes[2]:
        return 3
    elif x <= bin_sizes[3]:
        return 4
    elif x <= bin_sizes[4]:
        return 5
    elif x <= bin_sizes[5]:
        return 6
    elif x <= bin_sizes[6]:
        return 7
    else:
        return 8
    
data_long["binned_DEGs"] = data_long["DEGs"].apply(bin_DEG_numbers)    

In [61]:
all_normalized_distances_long = all_normalized_distances_mean.reset_index().rename(columns={'index': 'Cytokine'})
all_normalized_distances_long = all_normalized_distances_long.melt(id_vars=['Cytokine'], var_name='Cell_Type', value_name='Distance')

# clip at 0 and 1 (should basically be already the case)
all_normalized_distances_long["Distance"] = all_normalized_distances_long["Distance"].clip(lower=0, upper=1)

# merge with the DEGs
data_long = data_long.merge(all_normalized_distances_long, on=["Cytokine", "Cell_Type"])

In [62]:
def plot_overview_labels_along_y_axis(fig, ax, labels, vals_per_label, x_offset, no_y_vals,
                                      fontsize=7, fontweight="bold", color="black", linewidth=1, stagger=0,
                                      rotation=90):
    # Get the y-axis tick positions
    y_ticks = ax.get_yticks()
    
    # Get the first and last tick positions
    y_top = y_ticks[-1]
    y_bottom = y_ticks[0]
    
    total_distance = y_top - y_bottom
    step_per_label = total_distance / (no_y_vals - 1)
    
    # Freeze the current plot limits
    xlim = ax.get_xlim()
    ylim = ax.get_ylim()
    
    cumulative_vals = 0
    for index, label in enumerate(labels):
        y_pos = y_top - step_per_label * (cumulative_vals + vals_per_label[label]/2 - 0.5)
        curr_x_val = x_offset + stagger * (index % 2)
        
        if not "empty" in label and vals_per_label[label] > 0:
            # Add text annotations
            ax.text(
                curr_x_val,
                y_pos,
                label,
                ha='right',
                va='center',
                fontsize=fontsize,
                fontweight=fontweight,
                color=color,
                rotation=rotation,
                transform=ax.transData
            )
            
            adjustment_for_single_value = 0.4 if vals_per_label[label] == 1 else 0
            
            # Draw vertical line
            ax.plot(
                [curr_x_val + 0.2, curr_x_val + 0.2],
                [y_top - step_per_label * (cumulative_vals) + adjustment_for_single_value, 
                y_top - step_per_label * (cumulative_vals + vals_per_label[label] - 1) - adjustment_for_single_value],
                color=color,
                linewidth=linewidth,
                clip_on=False,
                zorder=0  # Ensures it doesn't interfere with other plot elements
            )
        
        cumulative_vals += vals_per_label[label]
    
    # Restore original plot limits to prevent them from changing
    ax.set_xlim(xlim)
    ax.set_ylim(ylim)

#### Plotting of absolute DEG numbers

In [ ]:
fig, ax = plt.subplots(figsize=cm_to_inches((5, 20)))

# Scatter plot with square markers; `s` controls the marker size, which we scale here by 'DEGs' values
for cell_type in ALL_CELL_TYPES:
    subset = data_long[data_long['Cell_Type'] == cell_type]
    ax.scatter(
        subset['Cell_Type'], subset['Cytokine'],
        s=subset['binned_DEGs'] * 2,  # Scale sizes as needed
        marker='s', c=[CELL_TYPE_COLORS[cell_type]], edgecolor='k',
        label=cell_type,
        alpha=subset['Distance']  # Scale alpha by 'Distance' values
    )

x_offset_level1 = -6  # Horizontal placement offset for family labels
vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()
plot_overview_labels_along_y_axis(fig, ax, SINGLE_FAMILY_ORDER, vals_per_label_level1, x_offset_level1,
                                  len(subset['Cytokine']),
                                  fontsize=6, fontweight="bold", color="black", linewidth=1,
                                  rotation=0)

# x_offset_level2 = -5  # Horizontal placement offset for family labels
# vals_per_label_level2 = cytokine_info["level2_family"].value_counts()
# plot_overview_labels_along_y_axis(fig, ax, level2_family_labels, vals_per_label_level2, x_offset_level2,
#                                   len(subset['Cytokine']),
#                                   fontsize=6, fontweight="normal", color="gray", linewidth=0.5)


# Create legend for DEG numbers
legend_elements = []
for bin_size, bin_label in zip(bin_sizes, bin_text):
    legend_elements.append(
        plt.scatter([], [], 
                   s=bin_DEG_numbers(bin_size) * 2,  # Base size matching the main plot
                   marker='s',
                   color='gray',
                   edgecolor='k',
                   label=f'{bin_label} DEGs')
    )

# Add the legend
leg = plt.legend(handles=legend_elements,
                bbox_to_anchor=(1.05, 1),
                loc='upper left',
                borderaxespad=0,
                fontsize=6)

# Adjust layout to make room for legend
plt.subplots_adjust(right=0.85)

# rotate x-axis labels
plt.xticks(rotation=90)
ax.set_ylim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(ax)
# set label font size to 6
plt.yticks(fontsize=5.5)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_DEG_numbers_by_cell_type.{format}"), 
                bbox_inches="tight", 
                dpi=300)

#### Plotting of relative DEG numbers

In [64]:
# pure maximum
data_long["max"] = data_long.groupby("Cell_Type")["DEGs"].transform("max")

# clip at at most 5% so that the dot size doesn't become too small
data_long["DEGs_clip"] = data_long.apply(lambda x: max(x["DEGs"], x["max"]/20.0), axis=1)
data_long["DEG_rel"] = data_long["DEGs_clip"] / data_long["max"]

# show the response magnitude as quintiles
data_long["distance_quintile"] = (pd.qcut(data_long["Distance"], q=5, labels=False) + 1) / 5

In [ ]:
# Plotting
fig, ax = plt.subplots(figsize=cm_to_inches((4.7, 20)))

# Scatter plot with square markers; `s` controls the marker size, which we scale here by 'DEGs' values
for cell_type in ALL_CELL_TYPES:
    subset = data_long[data_long['Cell_Type'] == cell_type]
    ax.scatter(
        subset['Cell_Type'], subset['Cytokine'],
        s=np.log10(subset['DEG_rel']+1) * 50,
        marker='s', c=[CELL_TYPE_COLORS[cell_type]], edgecolor='k',
        alpha=subset['distance_quintile'],
        linewidth=0.5
    )


x_offset_level1 = -6  # Horizontal placement offset for family labels
vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()

plot_overview_labels_along_y_axis(fig, ax, SINGLE_FAMILY_ORDER, vals_per_label_level1, x_offset_level1,
                                  len(subset['Cytokine']),
                                  fontsize=6, fontweight="bold", color="black", linewidth=1,
                                  rotation=0)

fig.canvas.draw()
ax.set_ylim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(ax)

plt.title(r"log$_{10}$(relative DEG number)",fontsize=7)
plt.xticks(rotation=90, fontsize=6)
# set label font size to 6
plt.yticks(fontsize=5.5)

# plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_relative_DEG_numbers_by_cell_type.{format}"), bbox_inches="tight", dpi=300)

#### Calculate and plot supplementary barplots (tsi, total DEGs, strong impact)

In [66]:
def calc_tsi(df, cytokine, column):
    cytokine_df = df[df["Cytokine"] == cytokine]
    values = cytokine_df[column].values
    max_value = np.max(values)
    tsi = sum((1-values/max_value)/(len(values)-1))
    return tsi

tsi_by_cytokine = {}
for cytokine in CYTOKINE_ORDER:
    tsi =  calc_tsi(data_long[data_long["Cell_Type"].isin(ALL_CELL_TYPES_EXCLUDE)], cytokine, "Distance")
    tsi_by_cytokine[cytokine] = tsi
    
tsi_by_cytokine = pd.Series(tsi_by_cytokine)    

In [ ]:
# need to invert the order due to the way that the heatmap is plotted
tsi_by_cytokine = tsi_by_cytokine[CYTOKINE_ORDER[::-1]]

# make a horizontal bar plot
plt.figure(figsize=cm_to_inches((1, 20)))
plt.barh(tsi_by_cytokine.index, tsi_by_cytokine.values, color="black")
plt.ylim(-1, len(CYTOKINE_ORDER))
# get the current figure and draw it (to get apply_greek_labels_heatmap to work)
fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())
plt.xticks(np.arange(0, 1.1, 1), [0, 1], rotation=0, fontsize=6)
plt.title("TSI", fontsize=7)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_tsi_by_cytokine.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
strong_impact_by_cytokine = strong_impact_by_cell_type.sum(axis=1)

# make a horizontal bar plot
plt.figure(figsize=cm_to_inches((1, 20)))
strong_impact_by_cytokine = strong_impact_by_cytokine[CYTOKINE_ORDER[::-1]]
plt.barh(strong_impact_by_cytokine.index, strong_impact_by_cytokine.values, color="black")
plt.ylim(-1, len(CYTOKINE_ORDER))
# get the current figure and draw it (to get apply_greek_labels_heatmap to work)
fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())
plt.xticks([0, len(ALL_CELL_TYPES)], [0, len(ALL_CELL_TYPES)], rotation=0, fontsize=6)
plt.title("strong\nimpact", fontsize=7)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_strong_impact_by_cytokine.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
# get the max DEGs per cell type
max_DEGs_by_cell_type = data_long.groupby("Cell_Type")["DEGs"].max()
max_DEGs_by_cell_type = max_DEGs_by_cell_type[ALL_CELL_TYPES]
# plot the max DEGs per cell type
plt.figure(figsize=cm_to_inches((5.36, 1.3)))
plt.bar(max_DEGs_by_cell_type.index, max_DEGs_by_cell_type.values, color="black", width=0.7)
plt.xticks([])
plt.ylabel("max.\nDEGs")
# cut whitespace at left and right
plt.xlim(-0.75, len(ALL_CELL_TYPES)-0.25)
plt.yticks([0, 10000], ["0", r"10$^{4}$"])
plt.ylim(0, 10000)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2_max_DEGs_by_cell_type.{format}"), bbox_inches="tight", dpi=300)


## 2.3 - Additional response magnitude plots

In [ ]:
strong_impact_by_cell_type_long = strong_impact_by_cell_type.melt(var_name="Cell_Type", value_name="strong_impact", ignore_index=False).reset_index().rename({"index": "Cytokine"}, axis=1)
data_long = pd.merge(data_long, strong_impact_by_cell_type_long, on=["Cell_Type", "Cytokine"])

In [119]:
alyssa_immunexpresso_df_binarized = alyssa_immunexpresso_df_binarized[alyssa_immunexpresso_df_binarized["Cell_Type"].isin(data_long["Cell_Type"].unique())]
alyssa_immunexpresso_df_binarized = alyssa_immunexpresso_df_binarized[alyssa_immunexpresso_df_binarized["Cytokine"].isin(data_long["Cytokine"].unique())]

In [122]:
data_long = pd.merge(data_long, alyssa_immunexpresso_df_binarized, on=["Cell_Type", "Cytokine"], how="left")

In [ ]:
data_long["xpresso"] = data_long["xpresso"].fillna(False)
data_long["xpresso"] = data_long["xpresso"].astype(bool)

In [ ]:
alyssa_immunexpresso_df_binarized[alyssa_immunexpresso_df_binarized["Cytokine"] == "4-1BBL"]

In [ ]:
# Plotting
fig, ax = plt.subplots(figsize=cm_to_inches((4.7, 20)))

# Scatter plot with square markers; `s` controls the marker size, which we scale here by 'DEGs' values
for cell_type in ALL_CELL_TYPES:
    subset = data_long[data_long['Cell_Type'] == cell_type]
    ax.scatter(
        subset['Cell_Type'], subset['Cytokine'],
        s = 10 * subset['strong_impact'].astype(int),
        marker='s', c="black", edgecolor="black", #c=[CELL_TYPE_COLORS[cell_type]], edgecolor='k',
        linewidth=0,
    )

    ax.scatter(
        subset['Cell_Type'], subset['Cytokine'],
        s = 10 * subset['xpresso'].astype(int),
        marker='s', facecolors="none",
        edgecolors='green', linewidth=1,
    )


x_offset_level1 = -6  # Horizontal placement offset for family labels
vals_per_label_level1 = cytokine_info_order["DEG_plot_family"].value_counts()

plot_overview_labels_along_y_axis(fig, ax, SINGLE_FAMILY_ORDER, vals_per_label_level1, x_offset_level1,
                                  len(subset['Cytokine']),
                                  fontsize=6, fontweight="bold", color="black", linewidth=1,
                                  rotation=0)

fig.canvas.draw()
ax.set_ylim(-1, len(CYTOKINE_ORDER))
apply_greek_labels_heatmap(ax)

# plt.title(r"log$_{10}$(relative DEG number)",fontsize=7)
plt.xticks(rotation=90, fontsize=6)
# set label font size to 6
plt.yticks(fontsize=5.5)

#plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"2.3_strong_impact_vs_expresso.{format}"), bbox_inches="tight", dpi=300)

# 3 - Literature responses
#### Receptor expression and expected responses

In [43]:
plot_folder = os.path.join(base_plot_folder, "3_literature_responses")
os.makedirs(plot_folder, exist_ok=True)

## 3.1 - Receptor expression

In [ ]:
def get_receptor_expression(average_control_expression, cytokine):
    receptor_genes_all = str(cytokine_info_parse.loc[cytokine, "receptor gene"])
    if receptor_genes_all == "nan":
        print(f"No receptor genes found for {cytokine}")
    
    all_receptor_expression_vals = []
    receptor_variants = receptor_genes_all.split(";")
    for receptor_variant in receptor_variants:
        receptor_genes = receptor_variant.split(",")
        receptor_genes = [gene.strip().replace(" ", "") for gene in receptor_genes]
        filtered_receptor_genes = [gene for gene in receptor_genes if gene in average_control_expression.index]
        
        if len(filtered_receptor_genes) > 0:
            receptor_expression = average_control_expression.loc[filtered_receptor_genes, :].min(axis=0) # all subunits must be expressed to trigger a response
            all_receptor_expression_vals.append(receptor_expression)
    if len(all_receptor_expression_vals) > 0:
        receptor_expression = pd.concat(all_receptor_expression_vals, axis=1).max(axis=1) # any of the alternative receptors can trigger a response
    else:
        receptor_expression = None
    return receptor_expression

receptor_expression_by_cytokine = {}
for cytokine in CYTOKINE_ORDER:
    receptor_expression = get_receptor_expression(control_norm_gene_count_df, cytokine)
    if receptor_expression is not None:
        receptor_expression_by_cytokine[cytokine] = receptor_expression
receptor_expression_by_cytokine = pd.DataFrame(receptor_expression_by_cytokine).T

#### Calculate relative receptor expression (with and without ILC+pDC (exclude)) for plotting purposes

In [ ]:
receptor_expression_by_cytokine_raw = receptor_expression_by_cytokine.copy()

# generate the relative receptor expression
receptor_expression_by_cytokine = receptor_expression_by_cytokine[ALL_CELL_TYPES]
receptor_expression_by_cytokine_relative = receptor_expression_by_cytokine.div(receptor_expression_by_cytokine[ALL_CELL_TYPES_EXCLUDE].max(axis=1), axis=0)

# generate the relative receptor expression (excluding ILC and pDC)
receptor_expression_by_cytokine_exclude = receptor_expression_by_cytokine.loc[:, ~receptor_expression_by_cytokine.columns.isin(["ILC", "pDC"])]
receptor_expression_by_cytokine_exclude_relative = receptor_expression_by_cytokine_exclude.div(receptor_expression_by_cytokine_exclude.max(axis=1), axis=0)

# we should have found receptors for all cytokines except for IL-32-beta
for cytokine in CYTOKINE_ORDER:
    if not cytokine in receptor_expression_by_cytokine_relative.index:
        print(f"No receptor expression found for {cytokine}")

# get the subset of distances relevant for later plotting
all_normalized_distances_mean_subset = all_normalized_distances_mean.loc[receptor_expression_by_cytokine_relative.index, receptor_expression_by_cytokine_relative.columns]
all_normalized_distances_mean_exclude_subset = all_normalized_distances_mean.loc[receptor_expression_by_cytokine_exclude_relative.index, receptor_expression_by_cytokine_exclude_relative.columns]

#### Scatter plot between the receptor expression and response magnitude/DEG number

In [46]:
# generate melted dataframes for the scatter plot
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative.melt(ignore_index=False).rename(columns={"value": "relative_expression"})
all_normalized_distances_mean_flat = all_normalized_distances_mean_subset.melt(ignore_index=False).rename(columns={"value": "distance"}).rename(columns={"variable": "cell_type"})
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.reset_index().merge(
    all_normalized_distances_mean_flat.reset_index(),
    on=["index", "cell_type"]).set_index("index")

# add the raw receptor expression
raw_expression = receptor_expression_by_cytokine_raw.melt(ignore_index=False)
raw_expression = raw_expression.rename(columns={"value": "raw_expression", "variable": "cell_type"})
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.merge(
    raw_expression, 
    left_on=["index", "cell_type"],
    right_on=[raw_expression.index, "cell_type"]
)

# add the relative number of DEGs
deg_count = data_long[["Cell_Type", "Cytokine", "DEG_rel"]].set_index("Cytokine").rename(columns={"Cell_Type": "cell_type"})
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.merge(
    deg_count, 
    left_on=["index", "cell_type"],
    right_on=[deg_count.index, "cell_type"]
)

# order values by ALL_CELL_TYPES
receptor_expression_by_cytokine_relative_flat = receptor_expression_by_cytokine_relative_flat.sort_values(by="cell_type", key=lambda x: pd.Series(x).map(dict(enumerate(ALL_CELL_TYPES))))

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=cm_to_inches((10, 10)), constrained_layout=True)
axes = axes.flatten()

# Collect legend labels
legend_labels = []

for i, cell_type in enumerate(ALL_CELL_TYPES):
    ax = axes[i]
    df_subset = receptor_expression_by_cytokine_relative_flat[receptor_expression_by_cytokine_relative_flat["cell_type"] == cell_type]
    r = np.corrcoef(np.log2(df_subset["raw_expression"]),
                    df_subset["distance"])[0, 1]
    scatter = ax.scatter(np.log2(df_subset["raw_expression"]),
                         df_subset["distance"], s=3,
                         color=CELL_TYPE_COLORS[cell_type])
    
    # Collect the label for the legend
    legend_labels.append((scatter, f"{cell_type} (r = {r:.2f})"))
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlim(-0.3, 9)
    
    if i % 4 == 0:  # First column
        ax.set_ylabel("response magnitude")
    if i >= 12:  # Last row
        ax.set_xlabel(r"$\log_{2}$(receptor rpm)")
    ax.label_outer()  # Only show outer labels

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Plot the legend once
handles, labels = zip(*legend_labels)
fig.legend(handles, labels, loc="lower right", fontsize=6, ncol=2)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_scatter_receptor_vs_distance.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=cm_to_inches((10, 10)), constrained_layout=True)
axes = axes.flatten()

# Collect legend labels
legend_labels = []

for i, cell_type in enumerate(ALL_CELL_TYPES):
    ax = axes[i]
    df_subset = receptor_expression_by_cytokine_relative_flat[receptor_expression_by_cytokine_relative_flat["cell_type"] == cell_type]
    r = np.corrcoef(np.log10(df_subset["raw_expression"]),
                    np.log10(df_subset["DEG_rel"]))[0, 1]
    scatter = ax.scatter(np.log10(df_subset["raw_expression"]),
                         np.log10(df_subset["DEG_rel"]), s=3,
                         color=CELL_TYPE_COLORS[cell_type])
    
    # Collect the label for the legend
    legend_labels.append((scatter, f"{cell_type} (r = {r:.2f})"))
    
    if i % 4 == 0:  # First column
        ax.set_ylabel("log$_{10}$(rel. DEGs)")
    if i >= 12:  # Last row
        ax.set_xlabel(r"$\log_{10}$(receptor rpm)")
    ax.label_outer()  # Only show outer labels

# Remove any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

# Plot the legend once
handles, labels = zip(*legend_labels)
fig.legend(handles, labels, loc="lower right", fontsize=6, ncol=2)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_scatter_receptor_vs_DEGs.{format}"), bbox_inches="tight", dpi=300)

### Expression heatmaps

In [ ]:
fig = plt.figure(figsize=cm_to_inches((6, 20)))
ax = sns.heatmap(np.log10(receptor_expression_by_cytokine_relative.clip(0.01, 1)), 
                cmap="coolwarm", 
                square=True, 
                cbar_kws={"shrink": 0.4, "label": r"$\log_{10}$(relative receptor expression)"})

# Explicitly set all y-ticks and labels
ax.set_yticks(np.arange(len(receptor_expression_by_cytokine_relative.index)) + 0.5)
ax.set_yticklabels([cytokine for cytokine in receptor_expression_by_cytokine_relative.index], fontsize=5.5)

# Apply Greek letter substitutions
apply_greek_labels_heatmap(ax)

# Adjust other plot parameters
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis='y', length=0)
ax.tick_params(axis='x', rotation=90, labelsize=6, length=0)
ax.set_title("relative\nreceptor expression")

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_receptor_expression_by_cytokine_relative_log.{format}"), 
                bbox_inches="tight", 
                dpi=300)

In [ ]:
max_receptor_expression = receptor_expression_by_cytokine[ALL_CELL_TYPES_EXCLUDE].max(axis=1)

# make a horizontal bar plot
plt.figure(figsize=cm_to_inches((1, 22.7)))
max_receptor_expression = max_receptor_expression[::-1]
plt.barh(y=max_receptor_expression.index, width=np.log10(max_receptor_expression.values), color="black", height=0.7)
plt.ylim(-0.5, len(max_receptor_expression)-0.5)
# get the current figure and draw it (to get apply_greek_labels_heatmap to work)
fig = plt.gcf()
fig.canvas.draw()
apply_greek_labels_heatmap(plt.gca())
plt.xlabel(r"$\log_{10}$(max. rpm)", fontsize=6)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_maximum_receptor_expression_by_cytokine.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
fig = plt.figure(figsize=cm_to_inches((6, 20)))
ax = sns.heatmap(np.log10(all_normalized_distances_mean_exclude_subset.clip(0.1)),
                cmap="rocket", 
                square=True, 
                cbar_kws={"shrink": 0.4, "label": r"$\log_{10}$(relative distance to control)"})   

# Explicitly set all y-ticks and labels
ax.set_yticks(np.arange(len(receptor_expression_by_cytokine_relative.index)) + 0.5)
ax.set_yticklabels([cytokine for cytokine in receptor_expression_by_cytokine_relative.index], fontsize=5.5)

# Apply Greek letter substitutions
apply_greek_labels_heatmap(ax)

# Adjust other plot parameters
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis='y', length=0)
ax.tick_params(axis='x', rotation=90, labelsize=6, length=0)
ax.set_title("relative distance\nto control")

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_distances_by_cytokine_relative.{format}"), 
                bbox_inches="tight", 
                dpi=300)

## 3.2 - Individual cell type and gene literature responses

In [52]:
plot_folder = os.path.join(base_plot_folder, "3.2_individual_literature_responses")
os.makedirs(plot_folder, exist_ok=True)

In [53]:
edger_df_padj_upregulated = edger_df_padj.copy()
edger_df_padj_downregulated = edger_df_padj.copy()

# Get boolean mask of negative fold changes
negative_fc_mask = edger_df_log2FoldChange < 0
positive_fc_mask = edger_df_log2FoldChange > 0

# Set padj values to 1 where fold change is negative
edger_df_padj_upregulated[negative_fc_mask] = 1
edger_df_padj_downregulated[positive_fc_mask] = 1

### Individual response lookup

In [54]:
# manually look up
strongest_response_in_literature = {}
upregulated_genes = pd.DataFrame(columns=["gene", "cell_type", "cytokine", "response_type"])

def evaluate_cytokine(current_cytokine, strongest_response_in_literature):
    data_long_cytokine = data_long[data_long["Cytokine"] == current_cytokine].copy().set_index("Cell_Type")
    print("There should be strong response in the following cell types:")
    print(strongest_response_in_literature[current_cytokine])
    print("The relative DEGs and Magnitudes are:")
    print(data_long_cytokine[["DEG_rel", "Distance"]])

def print_regulated_genes(upregulated_genes, current_cytokine):
    # Get genes for this cytokine
    cytokine_genes = upregulated_genes[upregulated_genes["cytokine"] == current_cytokine]
    
    # Group by cell type
    for cell_type, cell_genes in cytokine_genes.groupby("cell_type"):
        print(f"{cell_type}")
        print("--------------------------------")
        
        for _, row in cell_genes.iterrows():
            gene = row["gene"]
            if gene in edger_df_log2FoldChange.index:
                fc = edger_df_log2FoldChange.loc[gene, (cell_type, current_cytokine)]
                pval = edger_df_padj.loc[gene, (cell_type, current_cytokine)]
                
                print(f"The fold change for {gene} in response to {current_cytokine} is {fc}")
                print(f"The p-value for {gene} in response to {current_cytokine} is {pval:e}" \
                    if not pd.isna(pval) else f"The p-value for {gene} in response to {current_cytokine} is NaN")
                print("----")
            else:
                if gene in unfiltered_genes:
                    print(f"The gene {gene} is not in padj, but is in the list of unfiltered genes.")
                else:
                    print(f"The gene {gene} is not in padj, and is not in the list of unfiltered genes.")

#### We're interested in these (fairly randomly chosen) cytokines: IL-3, IL-32-beta, GM-CSF, TPO, IL-7, CD40L, IL-4, IL-12

In [ ]:
# --------------------------------------------------
# IL-3: 10.3389/fimmu.2024.1411047
current_cytokine = "IL-3"
print(f"Processing {current_cytokine}")
print("""CD123 (IL3RA) is mainly expressed on hematopoietic cells, including hematopoietic stem and progenitor cells (HSPCs),
basophils, eosinophils, mast cells, non-classical monocytes, macrophages, human plasmacytoid dendritic cells (pDCs)
and activated T and B cells (...) In steady-state, murine monocytes and human classical
monocytes do not express the receptor for IL-3. Only human intermediate and non-classical monocytes express it.""")

strongest_response_in_literature[current_cytokine] = ["HSPC", "CD16 Mono"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

print("The expression in CD14 Monos is curiously high.")

"""IL-3 stimulates the growth and
differentiation of HSPCs from bone marrow cultures and in naive or
sublethally irradiated mice into a large range of cell lineages,
including basophils, neutrophils, eosinophils, macrophages"""

"""IL-3 synergizes with IL-4 to enhance CCL17 expression in CD14+ monocytes """
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'CCL3', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'}, 
    {'gene': 'CCL7', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'},
    {'gene': 'CCL17', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'}
])])

print("upregulated genes")
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-32-beta: 10.1002/JLB.3MR0620-550R
current_cytokine = "IL-32-beta"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["NK CD56low", "NK CD56hi", "CD14 Mono", "CD16 Mono", "cDC"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# pick some upregulated cytokines
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'IL1B', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'inflammatory'},
    {'gene': 'IL6', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'inflammatory'}
])])

print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# M/GM-CSF: 10.1016_j.immuni.2016.10.026
current_cytokine = "GM-CSF"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["CD14 Mono", "CD16 Mono", "cDC"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# Croxford (2015) - Immunity - The Cytokine GM-CSF (pick a subset of genes for plotting)
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'PYCARD', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    #{'gene': 'MFGE8', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    #{'gene': 'CD1D', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    #{'gene': 'PLD1', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    #{'gene': 'SCARB1', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    {'gene': 'CLEC7A', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    #{'gene': 'ANXA1', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'phagocytosis'},
    {'gene': 'CCL17', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'},
    {'gene': 'CCL24', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'},
    #{'gene': 'CXCL3', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'chemotaxis'}
])])
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# TPO: 10.1080/09537104.2021.1925102
current_cytokine = "TPO"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["HSPC"] 
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# de Graaf (2011) - Cell cycle - Thrombopoietin and hematopoietic stem cells
# these genes aren't really mentioned directly, so don't plot this
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'CDKN1C', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'quiescence'},
    {'gene': 'CDKN2D', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'quiescence'},
    {'gene': 'HOXB4', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'HOXA5', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'HOXA9', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'HOXA10', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'stemness'},
    {'gene': 'BCL2L1', 'cytokine': current_cytokine, 'cell_type': 'HSPC', 'response_type': 'survival'}
])])

print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-7: 10.3389/fimmu.2021.747324
current_cytokine = "IL-7"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["CD4 T cell", "CD8 T cell", "MAIT", "NKT", "Treg"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# IL-7: 10.3389/fimmu.2021.747324
# Don't add CD14 Monos here to avoid clutter
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'BCL2', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'survival'},
    {'gene': 'PIM1', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'survival'},
    {'gene': 'BCL2', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'survival'},
    {'gene': 'PIM1', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'survival'},
    {'gene': 'IFNG', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytokine'},
    # {'gene': 'FASLG', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytokine'},
    # {'gene': 'CD40', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'activation'},
    # {'gene': 'CD80', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'activation'},
    # {'gene': 'HLA-DRA', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'activation'}
])])
print("upregulated genes")
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# CD40L: 10.1111/j.1600-065X.2009.00782.x
current_cytokine = "CD40L"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["B cell", "cDC", "CD14 Mono", "CD16 Mono"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# all of these work, but I'll pick two to actually plot
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'NFKB2', 'cytokine': current_cytokine, 'cell_type': 'B cell', 'response_type': 'NF-kappaB'},
    #{'gene': 'IKBKE', 'cytokine': current_cytokine, 'cell_type': 'B cell', 'response_type': 'NF-kappaB'},
    {'gene': 'TRAF1', 'cytokine': current_cytokine, 'cell_type': 'B cell', 'response_type': 'NF-kappaB'},
    #{'gene': 'UBE2K', 'cytokine': current_cytokine, 'cell_type': 'B cell', 'response_type': 'NF-kappaB'}
])])
print("upregulated genes")
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-4: 10.12703/r/10-71
current_cytokine = "IL-4"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["B cell", "CD4 T cell", "CD8 T cell", "MAIT", "NKT", "Treg", "NK CD56low", "NK CD56hi"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

# only plot B cell responses for this one
upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'FCER2', 'cytokine': current_cytokine, 'cell_type': 'B cell', 'response_type': 'isotype'}, # isotype switch, IgE receptor
    #{'gene': 'IGHG4', 'cytokine': current_cytokine, 'cell_type': 'B cell', 'response_type': 'isotype'},# isotype switch, IgG4 
    {'gene': 'IGHE', 'cytokine': current_cytokine, 'cell_type': 'B cell', 'response_type': 'isotype'}, # isotype switch, IgG3
    #{'gene': 'BCL2', 'cytokine': current_cytokine, 'cell_type': 'B cell', 'response_type': 'other'}, # prevents apoptosis
    {'gene': 'IL4I1', 'cytokine': current_cytokine, 'cell_type': 'B cell', 'response_type': 'other'}, ## of recent interest
    {'gene': 'IL4R', 'cytokine': current_cytokine, 'cell_type': 'B cell', 'response_type': 'other'}, ## induce its own receptor
    # {'gene': 'IL4R', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'other'}, ## induce its own receptor
    # {'gene': 'GATA3', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'differentiation'},
    # {'gene': 'IL13', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'cytokine production'},
    # {'gene': 'IL5', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'cytokine production'},
    # {'gene': 'IL1RL1', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'activation'},
    # {'gene': 'CD28', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'activation'},
    # {'gene': 'IL12RB2', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'receptor expression'},
    # {'gene': 'CD14', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'activation'},
    # {'gene': 'IL1B', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'cytokine production'},  #https://doi.org/10.1182/blood.V76.7.1392.1392
    # {'gene': 'TNF', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'cytokine production'},  #https://doi.org/10.1182/blood.V76.7.1392.1392
    # {'gene': 'IL6', 'cytokine': current_cytokine, 'cell_type': 'CD14 Mono', 'response_type': 'cytokine production'}  #https://doi.org/10.1182/blood.V76.7.1392.1392
])])
print("upregulated genes")
print_regulated_genes(upregulated_genes, current_cytokine)

In [ ]:
# --------------------------------------------------
# IL-12: 10.17179/excli2020-3104
current_cytokine = "IL-12"
print(f"Processing {current_cytokine}")

strongest_response_in_literature[current_cytokine] = ["NK CD56low", "NK CD56hi", "CD4 T cell", "CD8 T cell", "MAIT", "NKT", "Treg"]
evaluate_cytokine(current_cytokine, strongest_response_in_literature)

upregulated_genes = pd.concat([upregulated_genes, pd.DataFrame([
    {'gene': 'IFNG', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytokine'},
    {'gene': 'TNF', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytokine'},
    {'gene': 'IFNG', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'cytokine production'},
    #{'gene': 'TBX21', 'cytokine': current_cytokine, 'cell_type': 'CD4 T cell', 'response_type': 'cytokine production'}, 
    # {'gene': 'IFNG', 'cytokine': current_cytokine, 'cell_type': 'CD8 T cell', 'response_type': 'cytokine production'},
    {'gene': 'GZMB', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytotoxicity'},
    #{'gene': 'FASLG', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytotoxicity'},
    {'gene': 'PRF1', 'cytokine': current_cytokine, 'cell_type': 'NK CD56hi', 'response_type': 'cytotoxicity'}
])])
print("upregulated genes")
print_regulated_genes(upregulated_genes, current_cytokine)

### Check if expected global impact conforms to expectations

In [63]:
count_significant_responses = 0
expected_responses = pd.DataFrame(index=strongest_response_in_literature.keys(), columns=ALL_CELL_TYPES_EXCLUDE, data=0)
for cytokine in strongest_response_in_literature.keys():
    for cell_type in strongest_response_in_literature[cytokine]:
        if cell_type in ALL_CELL_TYPES_EXCLUDE:
            expected_responses.loc[cytokine, cell_type] = 1
            if strong_impact_by_cell_type.loc[cytokine, cell_type]:
                count_significant_responses += 1
        else:
            print(f"The cell type {cell_type} is not in the list of all cell types.")

In [ ]:
no_significant_responses_subset = strong_impact_by_cell_type.loc[expected_responses.index, expected_responses.columns][ALL_CELL_TYPES_EXCLUDE]
count_significant_responses / expected_responses.sum().sum()

In [65]:
all_relative_distances_mean_subset = all_relative_distances_mean.loc[expected_responses.index, expected_responses.columns][ALL_CELL_TYPES_EXCLUDE]
relative_degs_subset = data_long.pivot(index="Cytokine", columns="Cell_Type", values="DEG_rel").loc[expected_responses.index, expected_responses.columns][ALL_CELL_TYPES_EXCLUDE]
relative_degs_subset = relative_degs_subset.div(relative_degs_subset.max(axis=1), axis=0)

In [ ]:
plt.figure(figsize=cm_to_inches((5,3.8)))
ax = sns.heatmap(all_relative_distances_mean_subset, vmin=0.0, vmax=1.0, cmap="rocket",
            cbar_kws={"ticks": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0], "label": "rel. response magnitude", "shrink": 0.6}, square=True)
# add a green box around the expected responses
for cytokine_index, cytokine in enumerate(expected_responses.index):
    for cell_type_index, cell_type in enumerate(expected_responses.columns):
        if expected_responses.loc[cytokine, cell_type] == 1:
            if strong_impact_by_cell_type.loc[cytokine, cell_type]:
                rect = plt.Rectangle((expected_responses.columns.get_loc(cell_type),
                                expected_responses.index.get_loc(cytokine)),
                                1, 1, fill=False, edgecolor="lightgreen", linewidth=1.3, clip_on=False, zorder=2)
            else:
                rect = plt.Rectangle((expected_responses.columns.get_loc(cell_type),
                                expected_responses.index.get_loc(cytokine)),
                                1, 1, fill=False, edgecolor="coral", linewidth=1.3, clip_on=False, zorder=3)
            plt.gca().add_patch(rect)
            
# Add legend for green box
# Create custom legend patch for green box
green_box = plt.Rectangle((0,0), 1, 1, fill=False, edgecolor="lightgreen", linewidth=2)
plt.gca().add_artist(plt.legend([green_box], ['expected response'], loc='center left', bbox_to_anchor=(0.5, 0.5), handlelength=1, handletextpad=1))

apply_greek_labels_heatmap(ax)
plt.gca().tick_params(length=0)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_expected_strength_responses_literature.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
plt.figure(figsize=cm_to_inches((5,3.8)))
ax = sns.heatmap(relative_degs_subset, vmin=0.0, vmax=1.0, cmap="rocket",
            cbar_kws={"ticks": [0.0, 0.2, 0.4, 0.6, 0.8, 1.0], "label": "rel. DEGs", "shrink": 0.6}, square=True)
# add a green box around the expected responses
for cytokine_index, cytokine in enumerate(expected_responses.index):
    for cell_type_index, cell_type in enumerate(expected_responses.columns):
        if expected_responses.loc[cytokine, cell_type] == 1:
            rect = plt.Rectangle((expected_responses.columns.get_loc(cell_type),
                                expected_responses.index.get_loc(cytokine)),
                                1, 1, fill=False, edgecolor="lightgreen", linewidth=1.3, clip_on=False)
            plt.gca().add_patch(rect)
# Add legend for green box
# Create custom legend patch for green box
green_box = plt.Rectangle((0,0), 1, 1, fill=False, edgecolor="lightgreen", linewidth=2)
plt.gca().add_artist(plt.legend([green_box], ['expected response'], loc='center left', bbox_to_anchor=(0.5, 0.5), handlelength=1, handletextpad=1))

apply_greek_labels_heatmap(ax)
plt.gca().tick_params(length=0)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_expected_DEG_responses_literature.{format}"), bbox_inches="tight", dpi=300)

In [ ]:
plt.figure(figsize=cm_to_inches((5,3.8)))
receptor_expression_by_cytokine_relative_plot = receptor_expression_by_cytokine_relative.copy()
receptor_expression_by_cytokine_relative_plot.loc["IL-32-beta"] = np.nan
ax = sns.heatmap(np.log10(receptor_expression_by_cytokine_relative_plot.loc[expected_responses.index, expected_responses.columns].clip(0.01)), 
                cmap="coolwarm", 
                square=True, 
                cbar_kws={"shrink": 0.6, "label": r"$\log_{10}$(rel. receptor expr.)"})

# add a green box around the expected responses
for cytokine_index, cytokine in enumerate(expected_responses.index):
    if cytokine == "IL-32-beta":
        continue
    for cell_type_index, cell_type in enumerate(expected_responses.columns):
        if expected_responses.loc[cytokine, cell_type] == 1:
            if receptor_expression_by_cytokine_relative_plot.loc[cytokine, cell_type] > -1:
                rect = plt.Rectangle((expected_responses.columns.get_loc(cell_type),
                                    expected_responses.index.get_loc(cytokine)),
                                    1, 1, fill=False, edgecolor="lightgreen", linewidth=1.3, clip_on=False)
                plt.gca().add_patch(rect)
            else:
                rect = plt.Rectangle((expected_responses.columns.get_loc(cell_type),
                                    expected_responses.index.get_loc(cytokine)),
                                    1, 1, fill=False, edgecolor="coral", linewidth=1.3, clip_on=False)
                plt.gca().add_patch(rect)

apply_greek_labels_heatmap(ax)
plt.gca().tick_params(length=0)

for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"3_receptor_expression_subset.{format}"), 
                bbox_inches="tight", 
                dpi=300)

### Individual gene upregulation

In [69]:
upregulated_genes = upregulated_genes.reset_index(drop=True)
for index, row in upregulated_genes.iterrows():
    if row["gene"] in edger_df_log2FoldChange.index:
        fc = edger_df_log2FoldChange.loc[row["gene"], (row["cell_type"], row["cytokine"])]
        pval = edger_df_padj.loc[row["gene"], (row["cell_type"], row["cytokine"])]
        upregulated_genes.loc[index, "fc"] = fc
        upregulated_genes.loc[index, "pval"] = pval
upregulated_genes.dropna(inplace=True)
upregulated_genes["cyto_response"] = upregulated_genes["cytokine"] + "\n" + upregulated_genes["response_type"]

In [ ]:
cell_types = ["B cell", "CD4 T cell", "NK CD56hi", "CD14 Mono"]
for cell_type in cell_types:
    df_subset = upregulated_genes[upregulated_genes["cell_type"] == cell_type]

    # Fixed dimensions for the plot area (in cm)
    plot_width = len(df_subset['cyto_response'].unique())  # Width of plot area
    plot_height = 2  # Height of plot area
    margin_bottom = 2  # Extra space for x-labels
    margin_top = 0.5  # Extra space for title
    total_width = plot_width
    total_height = plot_height + margin_bottom + margin_top

    plt.figure(figsize=cm_to_inches((total_width, total_height)))

    # Create subplot with specific spacing to maintain fixed plot area
    plt.subplots_adjust(bottom=margin_bottom/total_height, 
                    top=1-margin_top/total_height)

    # Create colormap based on -log10(pvalue)
    log_pvals = -np.log10(df_subset['pval'].clip(1E-5))
    norm = plt.Normalize(0, log_pvals.max())
    cmap = plt.cm.viridis

    plt.scatter(df_subset['cyto_response'], df_subset['fc'], 
            c=log_pvals, cmap=cmap, norm=norm,
            alpha=1, s=10)

    # Add colorbar
    cbar = plt.colorbar()
    cbar.set_label('-log10(p-value)')

    # Add gene labels next to points
    for _, row in df_subset.iterrows():
        plt.annotate(row['gene'], 
                    (row['cyto_response'], row['fc']),
                    xytext=(5, 0), 
                    textcoords='offset points',
                    rotation=90,
                    fontsize=6)

    plt.xticks(rotation=90, va='top', ha='center')
    plt.ylabel(r'log$_2$(fold change)')
    plt.title(cell_type, fontsize=7, pad=10)
    plt.xlim(-0.25, len(df_subset['cyto_response'].unique()) - 0.75)
    plt.ylim(min(min(df_subset['fc']) - 0.5, -0.25), max(df_subset['fc']) + 0.5)

    apply_greek_labels_heatmap(plt.gca())

    # Remove top and right spines
    plt.gca().spines['top'].set_visible(False)
    plt.gca().spines['right'].set_visible(False)

    for format in ["png", "svg"]:
        plt.savefig(os.path.join(plot_folder, f"3_individual_upregulated_genes_by_cytokine_{cell_type}.{format}"), bbox_inches="tight", dpi=300)

# 4 - Average cytokine production response across cell types

Here, we
1) Get the average and max over all cytokines.
2) Filter to genes where the maximum average expression is at least 4 and the maximum maximum expression at least 20
    (This filters to cytokine genes that are expressed under at least one condition)
3) Filter the *mean* average expression to these columns.
4) Calculate the relative rpm across cell types

The resultant gene filter (expressed_genes_by_cell_type_dict) is later used to calculate correlation values between expression patterns.

In [257]:
plot_folder = os.path.join(base_plot_folder, "4_average_cytokine_production_by_cell_type")
os.makedirs(plot_folder, exist_ok=True)

In [258]:
# 1) Get the average and max over all cytokine stimulation conditions
sum_over_all_cytokines_mean = average_norm_sum_gene_count_df.T.groupby(level='cell_type').mean().T
sum_over_all_cytokines_max = average_norm_sum_gene_count_df.T.groupby(level='cell_type').max().T
substantial_detection = average_sum_gene_count_df.T.groupby(level=0).max().T # these are total counts

# 2) Calculate the per-gene filter for later use
expressed_genes_by_cell_type_df = (sum_over_all_cytokines_mean > 4) & (sum_over_all_cytokines_max > 20) & (substantial_detection > 20)
expressed_genes_by_cell_type_dict = {cell_type: list(expressed_genes_by_cell_type_df[cell_type][expressed_genes_by_cell_type_df[cell_type]].index) for cell_type in expressed_genes_by_cell_type_df.columns}

In [259]:
# 2) Calculate the per-cytokine filter (different from per-gene filter because cytokines are made up of multiple genes)

# First we calculate a proxy expression level for each cytokine given all genes that constitute it
cytokine_expression_df_mean = {}
cytokine_expression_df_max = {}
cytokine_expression_df_substantial = {}

# iterate over each cytokine gene individually
for index, row in cytokine_info.iterrows():
    cytokine_genes = row["gene"].split(",")
    cytokine_genes = [gene.strip() for gene in cytokine_genes]

    cytokine_genes = [gene for gene in cytokine_genes if gene in sum_over_all_cytokines_mean.index]
    if len(cytokine_genes) == 0:
        continue
    
    # the minimum is taken here under the assumption that all of the individual genes that make up a cytokine need to be expressed
    # e.g. IL23A, IL12B for IL-23
    cytokine_expression_df_mean[index] = sum_over_all_cytokines_mean.loc[cytokine_genes, :].min(axis=0)
    cytokine_expression_df_max[index] = sum_over_all_cytokines_max.loc[cytokine_genes, :].min(axis=0)
    cytokine_expression_df_substantial[index] = substantial_detection.loc[cytokine_genes, :].min(axis=0)
    
cytokine_expression_df_mean = pd.DataFrame(cytokine_expression_df_mean)
cytokine_expression_df_max = pd.DataFrame(cytokine_expression_df_max)
cytokine_expression_df_substantial = pd.DataFrame(cytokine_expression_df_substantial)

In [260]:
# 2) Get the actual expression filter for all cytokines and out tested (parse) cytokines
expressed_cytokines_by_cell_type_all_df = (cytokine_expression_df_mean > 4) & (cytokine_expression_df_max > 20) & (cytokine_expression_df_substantial > 20)
expressed_cytokines_by_cell_type_all = {}
for cell_type in expressed_cytokines_by_cell_type_all_df.index:
    series = expressed_cytokines_by_cell_type_all_df.loc[cell_type]
    expressed_cytokines_by_cell_type_all[cell_type] = [cyto for cyto in series.index if series[cyto]]
    
expressed_cytokines_by_cell_type_parse = {}
for cell_type in expressed_cytokines_by_cell_type_all.keys():
    curr_list = expressed_cytokines_by_cell_type_all[cell_type]
    expressed_cytokines_by_cell_type_parse[cell_type] = [cyto for cyto in curr_list if cyto in CYTOKINE_ORDER]

In [ ]:
# Filter to genes where the maximum average expression (across cell types) is at least 4 and the maximum maximum expression (across cell types) at least 20
cytokine_expression_df_filtered_mean = cytokine_expression_df_mean.loc[:, cytokine_expression_df_mean.loc[ALL_CELL_TYPES_EXCLUDE].max(axis=0) > 4]
cytokine_expression_df_filtered_max = cytokine_expression_df_max.loc[:, cytokine_expression_df_max.loc[ALL_CELL_TYPES_EXCLUDE].max(axis=0) > 20]

# get the intersection of the two column sets
columns_mean = cytokine_expression_df_filtered_mean.columns
columns_max = cytokine_expression_df_filtered_max.columns
columns_intersection = columns_mean.intersection(columns_max)
print(len(columns_intersection))

# 3) Filter the *mean* average expression to these columns.
cytokine_expression_df_filtered = cytokine_expression_df_filtered_mean.loc[ALL_CELL_TYPES_EXCLUDE, columns_intersection]

#### Calculate family counts

In [263]:
maximum_cell_type_count = {cell_type: 0 for cell_type in ALL_CELL_TYPES_EXCLUDE}
maximum_cell_type_identity = {}
maximum_cell_type_list = []
for column in cytokine_expression_df_filtered.columns:
    maximum_cell_type = cytokine_expression_df_filtered[column].idxmax()
    row = {
        "cell_type": maximum_cell_type,
        "cytokine": column
    }
    maximum_cell_type_list.append(row)
    maximum_cell_type_count[maximum_cell_type] += 1
    maximum_cell_type_identity[column] = maximum_cell_type

maximum_cell_type_df = pd.DataFrame(maximum_cell_type_list)
maximum_cell_type_df["family"] = maximum_cell_type_df["cytokine"].map(cytokine_name_to_level_2_family)
family_value_counts_df = maximum_cell_type_df.groupby("cell_type")["family"].value_counts().unstack().fillna(0)
# Order the family_value_counts_df by ALL_CELL_TYPES
family_value_counts_df = family_value_counts_df.reindex(ALL_CELL_TYPES_EXCLUDE).dropna()

available_families = family_value_counts_df.columns
ordered_available_families = [family for family in cytokine_level_2_order if family in available_families]
family_value_counts_df = family_value_counts_df[ordered_available_families]
family_value_counts_df = family_value_counts_df.iloc[:, ::-1]

In [ ]:
max_value_counts = maximum_cell_type_df["cell_type"].value_counts()
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    if not cell_type in max_value_counts.index:
        max_value_counts[cell_type] = 0
max_value_counts = max_value_counts[ALL_CELL_TYPES_EXCLUDE]

plt.figure(figsize=cm_to_inches((4.2,3.2)))
plt.bar(max_value_counts.index, max_value_counts.values, color="black", width=0.65)
plt.title("count strongest production", fontsize=7)
plt.xticks(rotation=90)
plt.xlim(-0.5, len(max_value_counts)-0.5)
plt.ylim(0, 30)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4_max_cell_type_count.{format}"), dpi=300)

In [ ]:
max_value_counts

In [ ]:
# Transpose the dataframe to plot by cytokine family instead of cell type
family_counts_by_cell_type = family_value_counts_df.T

# Plot the stacked vertical bar graph using matplotlib
fig, ax = plt.subplots(figsize=cm_to_inches((12, 8)))

# Plot each cell type as a separate bar segment
bottom = np.zeros(len(family_counts_by_cell_type.index))
for cell_type in family_counts_by_cell_type.columns:
    ax.barh(family_counts_by_cell_type.index, family_counts_by_cell_type[cell_type], 
           left=bottom, label=cell_type, color=CELL_TYPE_COLORS.get(cell_type, 'gray'))
    bottom += family_counts_by_cell_type[cell_type]

# Add labels and title
ax.set_xlabel('Maximum production per family')
ax.set_ylabel('')
ax.legend(title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
plt.xlim(0, 15)
plt.tight_layout()

# Show the plot
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"4_maximum_expression_by_family.{format}"), dpi=300)

In [78]:
# 4) Calculate the relative rpm across cell types
cytokine_expression_df_rel = cytokine_expression_df_filtered.div(cytokine_expression_df_filtered.max(axis=0), axis=1)

# sort the columns by the maximum cell type identity
maximum_cell_type_identity = pd.DataFrame([(k,v) for k,v in maximum_cell_type_identity.items()], columns=['gene', 'cell_type'])

# sort cell type by ALL_CELL_TYPES_EXCLUDE
maximum_cell_type_identity["cell_type"] = pd.Categorical(maximum_cell_type_identity["cell_type"], categories=ALL_CELL_TYPES_EXCLUDE, ordered=True)
maximum_cell_type_identity = maximum_cell_type_identity.sort_values(by="cell_type")

# apply sorting
cytokine_expression_df_rel = cytokine_expression_df_rel.loc[ALL_CELL_TYPES_EXCLUDE, maximum_cell_type_identity["gene"]]

In [ ]:
# plot in two separate groups
cytokine_groups_here = [['interleukin', 'interferon', 'TNF family', 'complement', 'growth factor', 'other'], ["chemokine"]]
for group_index, family in enumerate(cytokine_groups_here):
    cytokine_expression_df_rel_family = cytokine_expression_df_rel.loc[:, cytokine_info["family_level_1"].isin(family)]
    if group_index == 0:
        plt.figure(figsize=cm_to_inches((6, 16)))
    else:
        plt.figure(figsize=cm_to_inches((6, 10)))

    # Create heatmap
    ax = sns.heatmap(cytokine_expression_df_rel_family.T, cmap="Grays", vmin=0, vmax=1,
                     cbar_kws={"ticks": [0, 0.2, 0.4, 0.6, 0.8, 1], 
                              "label": "relative mean production", 
                              "shrink": 0.25}, 
                     square=False)

    # Fix y-axis labels
    plt.yticks(rotation=0)  # Make labels horizontal
    ax.tick_params(axis='y', labelsize=6)
    
    # Get all tick positions and labels
    tick_positions = np.arange(len(cytokine_expression_df_rel_family.columns))+0.5
    labels = [introduce_greek_letters(label) for label in cytokine_expression_df_rel_family.columns]
    
    # Set all ticks and labels explicitly
    ax.set_yticks(tick_positions)
    ax.set_yticklabels(labels)
    
    # set lengths to 0
    ax.tick_params(axis='y', labelsize=6, length=0)
    ax.tick_params(axis='x', length=0)

    # Remove x label
    plt.xlabel("")
    
    # Adjust layout
    plt.tight_layout()
    
    # Save plots
    for format in ["png", "svg"]:
        plt.savefig(os.path.join(plot_folder, f"4_average_cytokine_production_by_cell_type_{group_index}.{format}"), 
                    bbox_inches="tight",
                    dpi=300)

# 5 - Signaling Crosstalk

## 5.1 - Cytokine to cytokine crosstalk

In [299]:
plot_folder = os.path.join(base_plot_folder, "5.1_cytokine_to_cytokine_production")
os.makedirs(plot_folder, exist_ok=True)

In [300]:
# constrain to all cytokines that have a calculated fold change for all subunits (here as gene lists per cytokine)
all_cytokines_in_measured_genes = [cytokine for cytokine in cytokine_info["gene"] if all([subunit in edger_df_padj.index for subunit in cytokine.split(", ")])]

# get the padj and log2FoldChange values for all necessary genes
all_cytokines_in_measured_genes_flat = [subunit for cytokine in all_cytokines_in_measured_genes for subunit in cytokine.split(", ")]
edger_df_padj_cytokine = edger_df_padj.loc[edger_df_padj.index.isin(all_cytokines_in_measured_genes_flat), :].fillna(1)
edger_df_log2FoldChange_cytokine = edger_df_log2FoldChange.loc[edger_df_padj.index.isin(all_cytokines_in_measured_genes_flat), :].fillna(0)

In [ ]:
cytokine_response_df_all = pd.DataFrame(columns=["cell_type", "treatment", "response", "p_value", "fold_changes"])

for cell_type in ALL_CELL_TYPES:
    for cytokine_treatment in CYTOKINE_ORDER:
        for cytokine_response in all_cytokines_in_measured_genes:
            individual_cytokines = cytokine_response.split(", ")
            
            # skip this response if not significant
            skip = False
            for individual_cytokine in individual_cytokines:
                if not individual_cytokine in edger_df_padj_cytokine.index:
                    skip = True
            if skip:
                continue
            
            p_value = max([edger_df_padj_cytokine.loc[individual_cytokine, (cell_type, cytokine_treatment)].astype(float) for individual_cytokine in individual_cytokines])
            fold_changes = [edger_df_log2FoldChange_cytokine.loc[individual_cytokine, (cell_type, cytokine_treatment)].astype(float) for individual_cytokine in individual_cytokines]
            if p_value > 0.05:
                continue
            
            new_row = pd.DataFrame({
                "cell_type": [cell_type], 
                "treatment": [cytokine_treatment], 
                "response": [cytokine_response],
                "p_value": [p_value],
                "fold_changes": [fold_changes]
            })
            cytokine_response_df_all = pd.concat([cytokine_response_df_all, new_row], axis=0)

In [ ]:
# convert back to actual cytokine names
cytokine_translation = cytokine_info.reset_index().set_index("gene")["name"].to_dict()
cytokine_response_df_all.loc[:, "response"] = cytokine_response_df_all["response"].map(cytokine_translation)

# display duplicates
# cytokine_response_df_all[cytokine_response_df_all.duplicated(subset=["cell_type", "treatment", "response"], keep=False)]

# remove duplicates (basically just LTA, LTB because they have the same genes)
print(len(cytokine_response_df_all))
cytokine_response_df_all = cytokine_response_df_all.drop_duplicates(ignore_index=True, subset=["cell_type", "treatment", "response"])
print(len(cytokine_response_df_all))

In [303]:
# add the direction of the regulation
direction = []
for index, row in cytokine_response_df_all.iterrows():
    if all([fc > 1 for fc in row["fold_changes"]]):
        direction.append("upregulated")
    elif all([fc < -1 for fc in row["fold_changes"]]):
        direction.append("downregulated")
    else:
        direction.append("neither")
        
cytokine_response_df_all["direction"] = direction

In [304]:
# save the results
cytokine_response_df_all.to_csv(os.path.join(output_tables_folder, "5.1_cytokine-cytokine-crosstalk-all.csv"), index=False)

# filter to parse cytokines and save that subset
cytokine_response_df_all_filter = cytokine_response_df_all[cytokine_response_df_all["response"].isin(cytokine_info_parse.index)]
cytokine_response_df_all_filter.to_csv(os.path.join(output_tables_folder, "5.1_cytokine-cytokine-crosstalk-parse.csv"), index=False)

In [305]:
def create_sankey_diagram(df, column1_name=None, column1_order=None, column2_name=None, column2_order=None,
                          column3_name=None, column3_order=None,
                          color1 = "rgba(31, 119, 180, 0.8)", color2 = "rgba(44, 160, 44, 0.8)", color3 = "rgba(255, 127, 14, 0.8)",
                          title="Cytokine crosstalk between treatments and responses"):
    """
    Create a Sankey diagram with preserved node ordering and proper column spacing.
    
    Parameters:
    df: pandas DataFrame
    column1_name: name of the first column
    column1_order: list of column1 names in desired order (optional)
    column2_name: name of the second column
    column2_order: list of column2 names in desired order (optional)
    column3_name: name of the third column
    column3_order: list of column3 names in desired order (optional)
    """

    # get order of values in the columns
    column1_values = pd.unique(df[column1_name]).tolist()
    column2_values = pd.unique(df[column2_name]).tolist()
    column3_values = pd.unique(df[column3_name]).tolist()
    
    # Use provided orders or by the order of the values in the dataframe
    if column1_order is not None:
        missing_column1_values = set(df[column1_name].unique()) - set(column1_order)
        if missing_column1_values:
            raise ValueError(f"Column1 order missing values: {missing_column1_values}")
        column1_values = [value for value in column1_order if value in df[column1_name].unique()]
        
    if column2_order is not None:
        missing_column2_values = set(df[column2_name].unique()) - set(column2_order)
        if missing_column2_values:
            raise ValueError(f"Column2 order missing values: {missing_column2_values}")
        column2_values = [value for value in column2_order if value in df[column2_name].unique()]
        
    if column3_order is not None:
        missing_column3_values = set(df[column3_name].unique()) - set(column3_order)
        if missing_column3_values:
            raise ValueError(f"Column3 order missing values: {missing_column3_values}")
        column3_values = [value for value in column3_order if value in df[column3_name].unique()]

    # Create mapping dictionaries with ordered indices
    column1_to_idx = {t: idx for idx, t in enumerate(column1_values)}
    column2_to_idx = {c: idx + len(column1_values) for idx, c in enumerate(column2_values)}
    column3_to_idx = {r: idx + len(column1_values) + len(column2_values) for idx, r in enumerate(column3_values)}
    
    # Calculate link values while preserving order
    links = []
    
    # First column to second column (treatments to cell types)
    for t in column1_values:
        for c in column2_values:
            count = len(df[(df[column1_name] == t) & (df[column2_name] == c)])
            if count > 0:
                links.append({
                    'source': column1_to_idx[t],
                    'target': column2_to_idx[c],
                    'value': count
                })
        
    # Second column to third column (cell types to responses)
    for c in column2_values:
        for r in column3_values:
            count = len(df[(df[column2_name] == c) & (df[column3_name] == r)])
            if count > 0:
                links.append({
                    'source': column2_to_idx[c],
                    'target': column3_to_idx[r],
                    'value': count
                })

    # Extract sources, targets, and values from links
    sources = [link['source'] for link in links]
    targets = [link['target'] for link in links]
    values = [link['value'] for link in links]

    # Create color scheme
    n_column1_values = len(column1_values)
    n_column2_values = len(column2_values)
    n_column3_values = len(column3_values)
    
    colors = (
        [color1] * n_column1_values +  # Blue for column1
        [color2] * n_column2_values +   # Green for column2
        [color3] * n_column3_values     # Orange for column3
    )
    
    # Create node labels in the correct order
    display_labels = column1_values + column2_values + column3_values
    
    # Calculate node positions
    if n_column1_values > n_column3_values:
        x_positions = (
            [0.0] * n_column1_values +  # First column
            [0.5] * n_column2_values +  # Middle column
            [1.0] * n_column3_values     # Last column
        )
    else:
        x_positions = (
            [0.0] * n_column1_values +  # First column
            [0.3] * n_column2_values +  # Middle column
            [1.0] * n_column3_values     # Last column
        )
    
    # Create y positions that spread nodes evenly within their columns
    y_column1_values = np.linspace(0.05, 0.95, n_column1_values) if n_column1_values > 1 else [0.5]
    y_column2_values = np.linspace(0.05, 0.95, n_column2_values) if n_column2_values > 1 else [0.5]
    y_column3_values = np.linspace(0.05, 0.95, n_column3_values) if n_column3_values > 1 else [0.5]
    
    y_positions = list(y_column1_values) + list(y_column2_values) + list(y_column3_values)
    
    # Create Sankey Diagram
    fig = go.Figure(data=[go.Sankey(
        node=dict(
            pad=5,
            thickness=10,
            line=dict(color='black', width=0.5),
            label=display_labels,
            color=colors,
            x=x_positions,
            y=y_positions,
        ),
        link=dict(
            source=sources,
            target=targets,
            value=values,
            hovertemplate='Source: %{source.label}<br>Target: %{target.label}<br>Value: %{value}<extra></extra>',
            line=dict(color='rgba(0,0,0,0)', width=0)  # Remove edge stroke
        ),
        arrangement='snap'
    )])
    
    # Dynamic height scaling and improved margins
    fig.update_layout(
        font_size=6,
        height=max(n_column1_values, n_column2_values, n_column3_values) * 12.5,  # Adjust node spacing
        width=125,
        margin=dict(
            l=10,  # Left margin
            r=10,  # Right margin
            t=25,  # Top margin
            b=25   # Bottom margin to prevent cutoff
        ),
        showlegend=False,
        # set the font color to black
        font=dict(color='black'),
        title=title
    )
    return fig

In [306]:
treatment_order = CYTOKINE_ORDER
response_order = CYTOKINE_ORDER

treatment_order = [introduce_greek_letters(treatment) for treatment in treatment_order]
response_order = [introduce_greek_letters(response) for response in response_order]

# make sure the fold-change is sufficiently large for all gene products
cytokine_response_df_all_curr = cytokine_response_df_all_filter[cytokine_response_df_all_filter["direction"] == "upregulated"]
cytokine_response_df_all_curr.loc[:, "response"] = cytokine_response_df_all_curr["response"].apply(lambda x: introduce_greek_letters(x))
cytokine_response_df_all_curr.loc[:, "treatment"] = cytokine_response_df_all_curr["treatment"].apply(lambda x: introduce_greek_letters(x))

# now, do this separately for every cytokine
current_plot_folder = os.path.join(plot_folder, "5.1_upregulates_others")
os.makedirs(current_plot_folder, exist_ok=True)
for current_treatment in cytokine_response_df_all_curr["treatment"].unique():
    current_df = cytokine_response_df_all_curr[cytokine_response_df_all_curr["treatment"] == current_treatment]
    if len(current_df) < 6:
        continue
    fig = create_sankey_diagram(
        df=current_df,
        column1_name="treatment",
        column1_order=treatment_order,
        column2_name="cell_type",
        column3_name="response",
        column3_order=response_order,
        color1 = "rgba(44, 160, 44, 0.8)",
        color2 = "rgba(31, 119, 180, 0.8)",
        color3 = "rgba(44, 160, 44, 0.8)",
        title=None
    )
    fig.write_image(os.path.join(current_plot_folder, f"5.1_cytokine_{unidecode(current_treatment)}_upregulates_others.svg"), engine="kaleido")
    fig.write_image(os.path.join(current_plot_folder, f"5.1_cytokine_{unidecode(current_treatment)}_upregulates_others.png"), engine="kaleido")

current_plot_folder = os.path.join(plot_folder, "5.1_is_upregulated_by_others")
os.makedirs(current_plot_folder, exist_ok=True)
for current_response in cytokine_response_df_all_curr["response"].unique():
    current_df = cytokine_response_df_all_curr[cytokine_response_df_all_curr["response"] == current_response]
    if len(current_df) < 6:
        continue
    fig = create_sankey_diagram(
        df=current_df,
        column1_name="treatment",
        column1_order=treatment_order,
        column2_name="cell_type",
        column3_name="response",
        column3_order=response_order,
        color1 = "rgba(44, 160, 44, 0.8)",
        color2 = "rgba(31, 119, 180, 0.8)",
        color3 = "rgba(44, 160, 44, 0.8)",
        title=None
    )
    fig.write_image(os.path.join(current_plot_folder, f"5.1_cytokine_{unidecode(current_response)}_is_upregulated_by_others.svg"), engine="kaleido")
    fig.write_image(os.path.join(current_plot_folder, f"5.1_cytokine_{unidecode(current_response)}_is_upregulated_by_others.png"), engine="kaleido", scale=4)
    
# make sure the fold-change is significant for all gene products
cytokine_response_df_all_curr = cytokine_response_df_all_filter[cytokine_response_df_all_filter["direction"] == "downregulated"]
cytokine_response_df_all_curr.loc[:, "response"] = cytokine_response_df_all_curr["response"].apply(lambda x: introduce_greek_letters(x))
cytokine_response_df_all_curr.loc[:, "treatment"] = cytokine_response_df_all_curr["treatment"].apply(lambda x: introduce_greek_letters(x))

# now, do this separately for every cytokine
current_plot_folder = os.path.join(plot_folder, "5.1_downregulates_others")
os.makedirs(current_plot_folder, exist_ok=True)
for current_treatment in cytokine_response_df_all_curr["treatment"].unique():
    current_df = cytokine_response_df_all_curr[cytokine_response_df_all_curr["treatment"] == current_treatment]
    if len(current_df) < 6:
        continue
    fig = create_sankey_diagram(
        df=current_df,
        column1_name="treatment",
        column1_order=treatment_order,
        column2_name="cell_type",
        column3_name="response",
        column3_order=response_order,
        color1 = "rgba(44, 160, 44, 0.8)",
        color2 = "rgba(31, 119, 180, 0.8)",
        color3 = "rgba(44, 160, 44, 0.8)",
        title=None
    )
    fig.write_image(os.path.join(current_plot_folder, f"5.1_cytokine_{unidecode(current_treatment)}_downregulates_others.svg"), engine="kaleido")
    fig.write_image(os.path.join(current_plot_folder, f"5.1_cytokine_{unidecode(current_treatment)}_downregulates_others.png"), engine="kaleido")

#### Overview barplots

In [ ]:
# Look at significant responses (positive and negative)
cytokine_response_df_all_current = cytokine_response_df_all[cytokine_response_df_all["direction"].isin(["upregulated", "downregulated"])]

# only plot those with at least 3 responses
treatment_counts = cytokine_response_df_all_current["treatment"].value_counts()
treatment_counts = treatment_counts[treatment_counts > 3]
response_counts = cytokine_response_df_all_current["response"].value_counts()
response_counts = response_counts[response_counts > 3]

plt.figure(figsize=cm_to_inches((8, 3)))
plt.bar(treatment_counts.index, treatment_counts.values, color='skyblue', edgecolor='black')
plt.xticks(rotation=90)
plt.ylabel('outgoing\nsignaling responses')
apply_greek_labels_heatmap(plt.gca())

# Adjust margins to reduce whitespace
plt.margins(x=0.02)
plt.xticks(fontsize=6)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "5.1_stimulation_counts.svg"), bbox_inches="tight")

plt.figure(figsize=cm_to_inches((12, 3)))
plt.bar(response_counts.index, response_counts.values, color='skyblue', edgecolor='black')
plt.xticks(rotation=90)
plt.ylabel('incoming\nsignaling responses')
apply_greek_labels_heatmap(plt.gca())

# Adjust margins to reduce whitespace
plt.margins(x=0.02)
plt.xticks(fontsize=5.5)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "5.1_response_counts.svg"), bbox_inches="tight")

## 5.2 - Cell type - cell type communication

In [308]:
plot_folder = os.path.join(base_plot_folder, "5.2_cell_cell_communication")
os.makedirs(plot_folder, exist_ok=True)

In [ ]:
# Get number of expressed tested cytokines for each cell type
num_cytokines = {cell_type: len(cytokines) for cell_type, cytokines in expressed_cytokines_by_cell_type_parse.items()}
num_cytokines = pd.Series(num_cytokines)[ALL_CELL_TYPES]

plt.figure(figsize=cm_to_inches((6,4)))
plt.bar(num_cytokines.index, num_cytokines.values, color='skyblue', edgecolor='black')
plt.xticks(rotation=90)
plt.ylabel('Number of expressed cytokines')
plt.title('Distribution of expressed cytokines across cell types')
plt.show()

In [310]:
cell_type_secretion_df = pd.DataFrame(columns=["origin_cell_type", "secretion"])
for origin_cell_type, expressed_cytokines in expressed_cytokines_by_cell_type_parse.items():
    for cytokine in expressed_cytokines:
        cell_type_secretion_df = pd.concat([cell_type_secretion_df,
                                            pd.DataFrame({"origin_cell_type": origin_cell_type, "secretion": cytokine}, index=[0])], axis=0)

In [311]:
# get strong impact cytokines by cell type
all_normalized_distances_high = strong_impact_by_cell_type.copy()
all_normalized_distances_high["cytokine"] = all_normalized_distances_high.index
all_normalized_distances_high = all_normalized_distances_high.melt(id_vars="cytokine", var_name="cell_type", value_name="significant")
all_normalized_distances_high = all_normalized_distances_high[all_normalized_distances_high["significant"]]

In [312]:
secretion_to_response_df = pd.DataFrame(columns=["origin_cell_type", "secretion", "response_cell_type"])
for index, row in cell_type_secretion_df.iterrows():
    secretion = row["secretion"]
    response_cell_types = all_normalized_distances_high[all_normalized_distances_high["cytokine"] == secretion]
    for index, response_row in response_cell_types.iterrows():
        secretion_to_response_df = pd.concat([secretion_to_response_df,
                                              pd.DataFrame({"origin_cell_type": row["origin_cell_type"], "secretion": secretion, "response_cell_type": response_row["cell_type"]}, index=[0])], axis=0)

# save the dataframe
secretion_to_response_df.to_csv(os.path.join(output_tables_folder, "5.2_cell_cell_communication_df.csv"), index=False)

In [313]:
secretion_order = CYTOKINE_ORDER
secretion_order = [introduce_greek_letters(secretion) for secretion in secretion_order]

secretion_to_response_df["secretion"] = secretion_to_response_df["secretion"].apply(lambda x: introduce_greek_letters(x))
cell_type_order = ALL_CELL_TYPES

# now, do this separately for every cytokine
for cell_type in secretion_to_response_df["origin_cell_type"].unique():
    current_df = secretion_to_response_df[secretion_to_response_df["origin_cell_type"] == cell_type]
    column1_name = "origin_cell_type"
    column2_name = "secretion"
    column3_name = "response_cell_type"
    fig = create_sankey_diagram(
        df=current_df,
        column1_name=column1_name,
        column1_order=cell_type_order,
        column2_name=column2_name,
        column2_order=secretion_order,
        column3_name=column3_name,
        column3_order=cell_type_order,
        color1 = "rgba(31, 119, 180, 0.8)",
        color2 = "rgba(44, 160, 44, 0.8)", 
        color3 = "rgba(31, 119, 180, 0.8)",
        title=None
    )

    fig.write_image(os.path.join(plot_folder, f"5.2_cell_type_{unidecode(cell_type)}_secretes.svg"), engine="kaleido", scale=1)
    fig.write_image(os.path.join(plot_folder, f"5.2_cell_type_{unidecode(cell_type)}_secretes.png"), engine="kaleido", scale=4)

In [ ]:
cell_cell_signaling_counts = secretion_to_response_df.groupby("origin_cell_type")["response_cell_type"].value_counts().unstack().fillna(0)

# Plot the stacked vertical bar graph using matplotlib
fig, ax = plt.subplots(figsize=cm_to_inches((12, 8)))

# Plot each cell type as a separate bar segment
bottom = np.zeros(len(cell_cell_signaling_counts.index))
for cell_type in cell_cell_signaling_counts.columns:
    ax.barh(cell_cell_signaling_counts.index, cell_cell_signaling_counts[cell_type], 
           left=bottom, label=cell_type, color=CELL_TYPE_COLORS.get(cell_type, 'gray'))
    bottom += cell_cell_signaling_counts[cell_type]

# Add labels and title
# ax.set_xlabel('Maximum production per family')
# ax.set_ylabel('')
ax.legend(title='Cell Type', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=45, ha='right')
# plt.xlim(0, 15)
plt.tight_layout()

# Show the plot
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"5.2_cell_cell_communication_counts.{format}"), dpi=300)

In [315]:
# I don't think this is needed anymore
# expressed_cytokines_by_cell_type_all_family = {}
# for cell_type in ALL_CELL_TYPES:
#     expressed_cytokines_by_cell_type_all_family[cell_type] = [cytokine_name_to_level_2_family[gene] for gene in expressed_cytokines_by_cell_type_all[cell_type]]

## 5.3 - Count regulation of cytokines from different families

In [316]:
plot_folder = os.path.join(base_plot_folder, "5.3_family_counts")
os.makedirs(plot_folder, exist_ok=True)

In [317]:
family_level = "level2"

if family_level == "level2":
    cytokine_response_df_all["family"] = cytokine_response_df_all["response"].map(cytokine_name_to_level_2_family)
elif family_level == "level1":
    cytokine_response_df_all["family"] = cytokine_response_df_all["response"].map(cytokine_name_to_level_1_family)
else:
    raise Exception("Invalid level choice!")

cytokine_response_df_all["response_plus_cell_type"] = cytokine_response_df_all["response"] + "_" + cytokine_response_df_all["cell_type"]

In [318]:
family_value_counts_positive = {}
for treatment in cytokine_response_df_all["treatment"].unique():
    response_df_filter = cytokine_response_df_all[cytokine_response_df_all["treatment"] == treatment]
    response_df_filter = response_df_filter[response_df_filter["fold_changes"].apply(lambda x: all(fc > 1 for fc in x))]

    # keep only one occurrence of each upregulated gene
    response_df_filter = response_df_filter.drop_duplicates(subset="response_plus_cell_type")
    # response_df_filter = response_df_filter.drop_duplicates(subset="response")
    family_value_counts_positive[treatment] = response_df_filter["family"].value_counts()
    
family_value_counts_positive = pd.DataFrame(family_value_counts_positive)
family_value_counts_positive.fillna(0, inplace=True)
# convert to int
family_value_counts_positive = family_value_counts_positive.astype(int)

family_value_counts_negative = {}
for treatment in cytokine_response_df_all["treatment"].unique():
    response_df_filter = cytokine_response_df_all[cytokine_response_df_all["treatment"] == treatment]
    response_df_filter = response_df_filter[response_df_filter["fold_changes"].apply(lambda x: all(fc < -1 for fc in x))]
    
    # keep only one occurrence of each downregulated gene
    response_df_filter = response_df_filter.drop_duplicates(subset="response_plus_cell_type")
    # response_df_filter = response_df_filter.drop_duplicates(subset="response")
    family_value_counts_negative[treatment] = response_df_filter["family"].value_counts()
    
family_value_counts_negative = pd.DataFrame(family_value_counts_negative)
family_value_counts_negative.fillna(0, inplace=True)
# convert to int
family_value_counts_negative = family_value_counts_negative.astype(int)

In [ ]:
# inspect the results manually
subset = cytokine_response_df_all[cytokine_response_df_all["family"] == "IL-10"]
subset = subset[subset["direction"] == "upregulated"]
subset["treatment"].value_counts()

In [320]:
# add 0 counts for all missing families
if family_level == "level2":
    family_value_counts_positive = family_value_counts_positive.reindex(cytokine_level_2_order, fill_value=0)
    family_value_counts_negative = family_value_counts_negative.reindex(cytokine_level_2_order, fill_value=0)
if family_level == "level1":
    family_value_counts_positive = family_value_counts_positive.reindex(cytokine_level_1_order, fill_value=0)
    family_value_counts_negative = family_value_counts_negative.reindex(cytokine_level_1_order, fill_value=0)

In [ ]:
# total counts by cytokine treatment
total_significant_interactions_cytokine = family_value_counts_positive + family_value_counts_negative
total_significant_interactions_cytokine = total_significant_interactions_cytokine.sum(axis=0)

plt.figure(figsize=cm_to_inches((4, 3)))
plt.hist(total_significant_interactions_cytokine, bins=np.arange(0, 250, 10))
plt.xlabel("significant changes")
plt.ylabel("stimulation conditions")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.3_total_significant_interactions_cytokine.svg"), bbox_inches="tight")

In [ ]:
# total counts by cytokine response family
total_significant_interactions_family = family_value_counts_positive + family_value_counts_negative
total_significant_interactions_family = total_significant_interactions_family.sum(axis=1)

plt.figure(figsize=cm_to_inches((4, 3)))
plt.hist(total_significant_interactions_family, bins=np.arange(0, 250, 10))
plt.xlabel("significant changes")
plt.ylabel("family response")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.3_total_significant_interactions_family.svg"), bbox_inches="tight")

In [323]:
# load cytokine groupings and other things calculated later (run later parts of this notebook starting at "6-" first if necessary)
with open("output/6_highly_similar_groups.pkl", "rb") as f:
    highly_similar_groups = pickle.load(f)
with open("output/6_all_communities_cyto_cyto.pkl", "rb") as f:
    all_communities = pickle.load(f)
    all_communities_flat = [item for sublist in all_communities for item in sublist]

In [324]:
filter_cytokine_interactions = total_significant_interactions_cytokine[total_significant_interactions_cytokine > 3]
filter_family_interactions = total_significant_interactions_family[total_significant_interactions_family > 3]
family_value_counts_positive = family_value_counts_positive.loc[filter_family_interactions.index, filter_cytokine_interactions.index]
family_value_counts_negative = family_value_counts_negative.loc[filter_family_interactions.index, filter_cytokine_interactions.index]

# order columns by all_known_cytokine_and_chemokine_names
family_value_counts_positive = family_value_counts_positive.reindex(columns=[col for col in all_communities_flat if col in family_value_counts_positive.columns])
family_value_counts_negative = family_value_counts_negative.reindex(columns=[col for col in all_communities_flat if col in family_value_counts_negative.columns])
family_value_counts_both = family_value_counts_positive - family_value_counts_negative
family_value_counts_abs = family_value_counts_positive + family_value_counts_negative

family_value_counts_positive_rel = family_value_counts_positive.divide(family_value_counts_positive.max(axis=1), axis=0)
family_value_counts_negative_rel = family_value_counts_negative.divide(family_value_counts_negative.max(axis=1), axis=0)
family_value_counts_both_rel = family_value_counts_both.divide(abs(family_value_counts_both).max(axis=1), axis=0)

# subset the communities
all_communities_subset = []
for community in all_communities:
    all_communities_subset.append([c for c in community if c in family_value_counts_positive.columns])
highly_similar_groups_subset = []
for community in highly_similar_groups:
    highly_similar_groups_subset.append([c for c in community if c in family_value_counts_positive.columns])

In [325]:
# This is done to plot overview labels on the y-axis
counts_per_level1_family = {}
for level2_family in family_value_counts_positive_rel.index:
    level1_family = level2_to_level1.get(level2_family)
    if level1_family is not None:
        counts_per_level1_family[level1_family] = counts_per_level1_family.get(level1_family, 0) + 1
        
# invert the order of the dictionary
counts_per_level1_family = dict(reversed(counts_per_level1_family.items()))        

In [ ]:
if family_level == "level2":
    fig = plt.figure(figsize=cm_to_inches((14, 8)))
else:
    fig = plt.figure(figsize=cm_to_inches((12, 10)))

ax = sns.heatmap(family_value_counts_positive_rel, vmin=0, vmax=1, cmap="Reds",
            cbar_kws={"label": "relative number of upregulated cytokines", "shrink": 0.6 if family_level == "level2" else 0.15}, square=True)

for community in all_communities_subset:
    if len(community) > 1:
        start = family_value_counts_positive_rel.columns.get_loc(community[0])
        end = family_value_counts_positive_rel.columns.get_loc(community[-1]) + 1
        ax.add_patch(plt.Rectangle((start, 0), end - start, len(family_value_counts_positive_rel.index), fill=False, edgecolor='black', linewidth=1.5, clip_on=False))

for group in highly_similar_groups_subset:
    if len(group) > 1:
        start = family_value_counts_positive_rel.columns.get_loc(group[0])
        end = family_value_counts_positive_rel.columns.get_loc(group[-1]) + 1
        ax.add_patch(plt.Rectangle((start, 0), end - start, len(family_value_counts_positive_rel.index), fill=False, edgecolor='skyblue', linewidth=1.5, ls="--", clip_on=False))

plot_overview_labels_along_y_axis(fig, ax, counts_per_level1_family.keys(),
                                  counts_per_level1_family, -10, len(family_value_counts_positive_rel.index), stagger=1.5)

plt.xlabel("cytokine treatment")
plt.ylabel("")
plt.gca().tick_params(axis='both', length=0)
apply_greek_labels_heatmap(plt.gca())
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.3_family_value_counts_positive_{family_level}.svg"), bbox_inches="tight")

In [ ]:
if family_level == "level2":
    fig = plt.figure(figsize=cm_to_inches((14, 8)))
else:
    fig = plt.figure(figsize=cm_to_inches((12, 10)))



ax = sns.heatmap(family_value_counts_negative_rel, vmin=0, vmax=1, cmap="Blues",
            cbar_kws={"label": "relative number of downregulated cytokines", "shrink": 0.6 if family_level == "level2" else 0.15}, square=True)


plot_overview_labels_along_y_axis(fig, ax, counts_per_level1_family.keys(),
                                  counts_per_level1_family, -10, len(family_value_counts_positive_rel.index), stagger=1.5)

for community in all_communities_subset:
    if len(community) > 1:
        start = family_value_counts_positive_rel.columns.get_loc(community[0])
        end = family_value_counts_positive_rel.columns.get_loc(community[-1]) + 1
        ax.add_patch(plt.Rectangle((start, 0), end - start, len(family_value_counts_positive_rel.index), fill=False, edgecolor='black', linewidth=1.5, clip_on=False))

for group in highly_similar_groups_subset:
    if len(group) > 1:
        start = family_value_counts_positive_rel.columns.get_loc(group[0])
        end = family_value_counts_positive_rel.columns.get_loc(group[-1]) + 1
        ax.add_patch(plt.Rectangle((start, 0), end - start, len(family_value_counts_positive_rel.index), fill=False, edgecolor='coral', linewidth=1.5, ls="--", clip_on=False))

plt.ylabel("")
apply_greek_labels_heatmap(plt.gca())

plt.xlabel("cytokine treatment")
plt.gca().tick_params(axis='both', length=0)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.3_family_value_counts_negative_{family_level}.svg"), bbox_inches="tight")


In [ ]:
if family_level == "level2":
    fig = plt.figure(figsize=cm_to_inches((14, 8)))
else:
    fig = plt.figure(figsize=cm_to_inches((12, 10)))

df = family_value_counts_both_rel

ax = sns.heatmap(df, vmin=df.min().min(), vmax=df.max().max(), cmap="coolwarm",
            cbar_kws={"label": "relative number of regulated cytokines", "shrink": 0.6 if family_level == "level2" else 0.15}, square=True)


plot_overview_labels_along_y_axis(fig, ax, counts_per_level1_family.keys(),
                                  counts_per_level1_family, -10, len(family_value_counts_positive_rel.index), stagger=1.5)

for community in all_communities_subset:
    if len(community) > 1:
        start = family_value_counts_positive_rel.columns.get_loc(community[0])
        end = family_value_counts_positive_rel.columns.get_loc(community[-1]) + 1
        ax.add_patch(plt.Rectangle((start, 0), end - start, len(family_value_counts_positive_rel.index), fill=False, edgecolor='black', linewidth=2, clip_on=False))

for group in highly_similar_groups_subset:
    if len(group) > 1:
        start = family_value_counts_positive_rel.columns.get_loc(group[0])
        end = family_value_counts_positive_rel.columns.get_loc(group[-1]) + 1
        ax.add_patch(plt.Rectangle((start, 0), end - start, len(family_value_counts_positive_rel.index), fill=False, edgecolor='lightblue', linewidth=1.5, ls="--", clip_on=False))

plt.ylabel("")
apply_greek_labels_heatmap(plt.gca())

plt.xlabel("cytokine treatment")
plt.gca().tick_params(axis='both', length=0)

plt.tight_layout()
plt.savefig(os.path.join(plot_folder, f"5.3_family_value_counts_both_{family_level}.svg"), bbox_inches="tight")

# 6 - Cytokine similarity

## 6.1 - Within cell type correlations

In [ ]:
# FOR DEBUGGING/TESTING THE SIMILARITY CODE
cell_type = "CD14 Mono"
all_inclusive_filter = expressed_genes_by_cell_type_dict[cell_type]
existing_fold_change_vals = edger_df_log2FoldChange[cell_type].dropna(how='all').index
all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in existing_fold_change_vals]
cell_type_log2 = edger_df_log2FoldChange.loc[all_inclusive_filter, cell_type]

cytokine1 = "GM-CSF"
cytokine2 = "IL-3"

cell_data_1 = cell_type_log2[cytokine1].dropna()
cell_data_2 = cell_type_log2[cytokine2].dropna()

common_index = cell_data_1.index.intersection(cell_data_2.index)
cell_data_1 = cell_data_1[common_index]
cell_data_2 = cell_data_2[common_index]

r = np.corrcoef(cell_data_1, cell_data_2)[0, 1]
r

In [111]:
plot_folder = os.path.join(base_plot_folder, "6.1_within_cell_similarity")
os.makedirs(plot_folder, exist_ok=True)

from scipy.spatial.distance import pdist, squareform

In [ ]:
cytokine_combs = list(combinations_with_replacement(CYTOKINE_ORDER, 2))

if os.path.exists(os.path.join("output", f"6.1_r_vals_by_cell_type.pkl")):
    with open(os.path.join("output", f"6.1_r_vals_by_cell_type.pkl"), "rb") as f:
        r_vals_by_cell_type = pickle.load(f)
    with open(os.path.join("output", f"6.1_cosine_distances_by_cell_type.pkl"), "rb") as f:
        cosine_distances_by_cell_type = pickle.load(f)
    with open(os.path.join("output", f"6.1_euclidean_distances_by_cell_type.pkl"), "rb") as f:
        euclidean_distances_by_cell_type = pickle.load(f)
else:
    r_vals_by_cell_type = {}
    cosine_distances_by_cell_type = {}
    euclidean_distances_by_cell_type = {}
    
    for cell_type in ALL_CELL_TYPES:
        r_vals_by_cytokine = pd.DataFrame(index=[cytokine for cytokine in CYTOKINE_ORDER],
                                             columns=[cytokine for cytokine in CYTOKINE_ORDER])
        cosine_distance_by_cytokine = pd.DataFrame(index=[cytokine for cytokine in CYTOKINE_ORDER],
                                             columns=[cytokine for cytokine in CYTOKINE_ORDER])
        print(cell_type)
        i = 0
        
        # first, use the pre-calculated filter based on expression values
        all_inclusive_filter = expressed_genes_by_cell_type_dict[cell_type]
        
        # then, subset to those where we actually have fold change values for this cell type
        existing_fold_change_vals = edger_df_log2FoldChange[cell_type].dropna(how='all').index
        all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in existing_fold_change_vals]
        
        print(len(all_inclusive_filter))
        cell_type_log2 = edger_df_log2FoldChange.loc[all_inclusive_filter, cell_type]
        
        # ------------------- review this ---------------------------
        # euclidean distance
        condensed_distances = pdist(cell_type_log2.T, metric="euclidean")
        euclidean_distance_matrix = squareform(condensed_distances)
        df = pd.DataFrame(euclidean_distance_matrix, index=cell_type_log2.columns, columns=cell_type_log2.columns)
        euclidean_distances_by_cell_type[cell_type] = df
        
        # cosine distance
        condensed_distances = pdist(cell_type_log2.T, metric="cosine")
        cosine_distance_matrix = squareform(condensed_distances)
        df = pd.DataFrame(cosine_distance_matrix, index=cell_type_log2.columns, columns=cell_type_log2.columns)
        cosine_distances_by_cell_type[cell_type] = df
        # -------------------------------------------------------------

        for cytokine1, cytokine2 in cytokine_combs:
            if cytokine1 == cytokine2:
                r_vals_by_cytokine.loc[cytokine1, cytokine2] = 1.0
                continue
            
            # get values for both cytokines and drop na
            cell_data_1 = cell_type_log2[cytokine1].dropna()
            cell_data_2 = cell_type_log2[cytokine2].dropna()

            if len(cell_data_1) == 0 or len(cell_data_2) == 0:
                r_vals_by_cytokine.loc[cytokine1, cytokine2] = np.nan
                continue
            
            # get the intersection of the two cytokines
            common_index = cell_data_1.index.intersection(cell_data_2.index)
            cell_data_1 = cell_data_1[common_index]
            cell_data_2 = cell_data_2[common_index]
            
            r = np.corrcoef(cell_data_1, cell_data_2)[0, 1]
            r_vals_by_cytokine.loc[cytokine1, cytokine2] = r
            
        # fill in the other half of the correlation matrix
        r_vals_by_cytokine = r_vals_by_cytokine.fillna(r_vals_by_cytokine.T).astype(float)
        r_vals_by_cell_type[cell_type] = r_vals_by_cytokine.copy()

os.makedirs("output", exist_ok=True)
with open(os.path.join("output", f"6.1_r_vals_by_cell_type.pkl"), "wb") as f:
    pickle.dump(r_vals_by_cell_type, f)
with open(os.path.join("output", f"6.1_cosine_distances_by_cell_type.pkl"), "wb") as f:
    pickle.dump(cosine_distances_by_cell_type, f)
with open(os.path.join("output", f"6.1_euclidean_distances_by_cell_type.pkl"), "wb") as f:
    pickle.dump(euclidean_distances_by_cell_type, f)

In [113]:
# Create multi-index DataFrame
index = pd.MultiIndex.from_product([CYTOKINE_ORDER, CYTOKINE_ORDER], names=['cytokine1', 'cytokine2'])
r_vals_by_cell_type_df = pd.DataFrame(index=index, columns=ALL_CELL_TYPES)
cosine_distances_by_cell_type_df = pd.DataFrame(index=index, columns=ALL_CELL_TYPES)
euclidean_distances_by_cell_type_df = pd.DataFrame(index=index, columns=ALL_CELL_TYPES)

# Fill the DataFrames
for cell_type in ALL_CELL_TYPES:
    r_matrix = r_vals_by_cell_type[cell_type]
    cos_matrix = cosine_distances_by_cell_type[cell_type]
    euc_matrix = euclidean_distances_by_cell_type[cell_type]
    
    for cyt1 in CYTOKINE_ORDER:
        for cyt2 in CYTOKINE_ORDER:
            r_vals_by_cell_type_df.loc[(cyt1, cyt2), cell_type] = r_matrix.loc[cyt1, cyt2]
            cosine_distances_by_cell_type_df.loc[(cyt1, cyt2), cell_type] = cos_matrix.loc[cyt1, cyt2]
            euclidean_distances_by_cell_type_df.loc[(cyt1, cyt2), cell_type] = euc_matrix.loc[cyt1, cyt2]

### explore how correlation values depend on the impact classification

Could add extra code here to quantify this more comprehensively

In [114]:
# Apply logical AND across the index to find cell types where both IL-3 and GM-CSF have significant impact
both_strong = strong_impact_by_cell_type.loc["IL-3"] & strong_impact_by_cell_type.loc["GM-CSF"]
both_strong_columns = both_strong[both_strong].index
not_both_columns = both_strong[~both_strong].index

In [ ]:
r_vals_by_cell_type_df.loc[("IL-3", "GM-CSF"), both_strong_columns].mean()

In [ ]:
r_vals_by_cell_type_df.loc[("IL-3", "GM-CSF"), not_both_columns].mean()

### create scatter plots

In [117]:
def get_lower_triangle(df):
    """Get values from the lower triangle of a DataFrame (excluding the diagonal)."""
    return df.where(np.tril(np.ones(df.shape), k=-1).astype(bool))

def find_pairs_thresh(r_vals_by_cytokine, threshold):
    pairs = []
    for cytokine1, cytokine2 in cytokine_combs:
        if threshold > 0:
            if r_vals_by_cytokine.loc[cytokine1, cytokine2] > threshold:
                if cytokine1 != cytokine2:
                    pairs.append((cytokine1, cytokine2))
        if threshold < 0:
            if r_vals_by_cytokine.loc[cytokine1, cytokine2] < threshold:
                if cytokine1 != cytokine2:
                    pairs.append((cytokine1, cytokine2))
    return pairs

This takes 5 min to run

In [118]:
all_pairs_df_high = []
for cell_type in ALL_CELL_TYPES:
    r_vals = r_vals_by_cell_type[cell_type]
    pairs = find_pairs_thresh(r_vals, threshold=0.8)
    # sort the pairs by name
    pairs = sorted(pairs, key=lambda x: (x[0], x[1]))
    for cytokine1, cytokine2 in pairs:
        new_row = {"cell_type": cell_type, "cytokine1": cytokine1, "cytokine2": cytokine2, "r": r_vals.loc[cytokine1, cytokine2]}
        new_row = pd.DataFrame([new_row])
        if len(new_row) > 0:
            all_pairs_df_high.append(new_row)
all_pairs_df_high = pd.concat(all_pairs_df_high, ignore_index=True)

all_pairs_df_low = []
for cell_type in ALL_CELL_TYPES:
    r_vals = r_vals_by_cell_type[cell_type]
    pairs = find_pairs_thresh(r_vals, threshold=-0.4)
    # sort the pairs by name
    pairs = sorted(pairs, key=lambda x: (x[0], x[1]))
    for cytokine1, cytokine2 in pairs:
        new_row = {"cell_type": cell_type, "cytokine1": cytokine1, "cytokine2": cytokine2, "r": r_vals.loc[cytokine1, cytokine2]}
        new_row = pd.DataFrame([new_row])
        if len(new_row) > 0:
            all_pairs_df_low.append(new_row)
all_pairs_df_low = pd.concat(all_pairs_df_low, ignore_index=True)
    
all_pairs_df_high["cytokine_pair"] = all_pairs_df_high["cytokine1"] + "_" + all_pairs_df_high["cytokine2"]
all_pairs_df_low["cytokine_pair"] = all_pairs_df_low["cytokine1"] + "_" + all_pairs_df_low["cytokine2"]       

In [ ]:
pairs_to_plot = [
    ("IL-15", "IL-2"),
    ("IL-4", "IL-13"),
    ("IL-15", "IL-10"),
    ("IL-3", "GM-CSF"),
    ("IL-15", "APRIL"),
    ("IL-15", "VEGF")
]

from scipy.stats import gaussian_kde
corr_plot_folder = os.path.join(plot_folder, "correlation_plots")
os.makedirs(corr_plot_folder, exist_ok=True)

density_plot_flag = False
        
# ------------------------- Large plot -------------------------
fig, axs = plt.subplots(nrows=4, ncols=6, figsize=cm_to_inches((14, 8)), sharex=True, sharey=True)
axs = axs.flatten()
counter = 0

for cell_type in ALL_CELL_TYPES:
    if not cell_type in ["B cell", "CD14 Mono", "cDC", "CD4 T cell"]:
        continue
    print(cell_type)
    
    r_vals = r_vals_by_cell_type[cell_type]
    #pairs = find_pairs_thresh(r_vals, threshold=-0.4)
    #print(cell_type, pairs)
    pairs = pairs_to_plot
    
    all_inclusive_filter = expressed_genes_by_cell_type_dict[cell_type]
    existing_fold_change_vals = edger_df_log2FoldChange[cell_type].dropna(how='all').index
    all_inclusive_filter = [gene for gene in all_inclusive_filter if gene in existing_fold_change_vals]
    print(len(all_inclusive_filter))
    cell_type_log2 = edger_df_log2FoldChange.loc[all_inclusive_filter, cell_type].dropna()
    
    for cytokine1, cytokine2 in pairs:
        x = cell_type_log2.loc[:, cytokine1]
        y = cell_type_log2.loc[:, cytokine2]
        r = np.corrcoef(x, y)[0, 1]
        
        # ------------------------- Small plot -------------------------
        plt.figure(figsize=cm_to_inches((3, 3)))
        plt.scatter(x, y, s = 1, edgecolor="none", alpha=0.5, rasterized=True, zorder=1, label=f"r={r:.2f}")
        
        
        plt.ylim(-4, 4)
        plt.xticks([-4, 0, 4], fontsize=6)
        plt.yticks([-4, 0, 4], fontsize=6) 

        plt.xlabel(f"log$_{2}$FC", fontsize=6)
        plt.ylabel(f"log$_{2}$FC", fontsize=6)
        
        plt.title(f'{cytokine1} vs {cytokine2}', fontsize=6.5)
        plt.legend(loc="upper left", fontsize=6)
        
        plt.tight_layout()
        for format in ["png", "svg"]:
            plt.savefig(os.path.join(corr_plot_folder, f"6.1_small_corr_{cell_type}_{cytokine1}_vs_{cytokine2}.{format}"),
                            bbox_inches="tight", dpi=400)
        plt.close()
        
        # ---- subplots of large plot ----
        ax = axs[counter]
        
        # create a cross at the origin
        ax.plot([-5, 5], [0, 0], color="black", linewidth=1, zorder=0, alpha=0.5)
        ax.plot([0, 0], [-5, 5], color="black", linewidth=1, zorder=0, alpha=0.5)
        ax.scatter(x, y, s = 1, edgecolor="none", alpha=0.5, rasterized=True, zorder=1)
        
        ax.set_xlim(-4, 4)
        ax.set_ylim(-4, 4)
        ax.set_xticks([-4, -2, 0, 2, 4])
        ax.set_yticks([-4, -2, 0, 2, 4]) 

        # ax.set_xlabel(f"log$_{2}$FC({introduce_greek_letters(cytokine1)})", fontsize=6)
        # ax.set_ylabel(f"log$_{2}$FC({introduce_greek_letters(cytokine2)})", fontsize=6)
        ax.set_xlabel(f"{introduce_greek_letters(cytokine1)}", fontsize=6)
        ax.set_ylabel(f"{introduce_greek_letters(cytokine2)}", fontsize=6)
        #ax.set_title(f'{cell_type}, r={r:.2f}', fontsize=6)
        ax.text(0.05 if "IL-10" not in [cytokine1, cytokine2] else 0.4, 0.95, f"r={r:.2f}", transform=ax.transAxes, 
                fontsize=6, verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'))
        
        # Set tick length to 0
        ax.tick_params(axis='both', which='both', length=0)
        
        counter += 1
        
        # Hide x and y tick labels for inner subplots
        # Only show y tick labels for the first column
        # Only show x tick labels for the last row
        if not ax.get_subplotspec().is_first_col():
            #ax.set_ylabel('')
            plt.setp(ax.get_yticklabels(), visible=False)
        
        if not ax.get_subplotspec().is_last_row():
            #ax.set_xlabel('')
            plt.setp(ax.get_xticklabels(), visible=False)

plt.tight_layout(pad=0.5, h_pad=0.5, w_pad=0.5)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(corr_plot_folder, f"6.1_large_corr_panel.{format}"),
                    bbox_inches="tight", dpi=400)
plt.close()


In [ ]:
# Create a subplot grid based on number of cell types
n_cell_types = len(r_vals_by_cell_type.keys())
n_rows = 3
n_cols = 5

fig, axes = plt.subplots(n_rows, n_cols, figsize=cm_to_inches((16, 8)))
axes = axes.flatten()

# Plot histogram for each cell type
for idx, (cell_type, ax) in enumerate(zip(r_vals_by_cell_type.keys(), axes)):
    df_subset = r_vals_by_cell_type[cell_type].copy()
    # set all diagonals to NaN
    df_subset.values[np.diag_indices_from(df_subset)] = np.nan
    flattened = pd.Series(df_subset.values.flatten()).dropna()
    ax.hist(flattened, bins=np.arange(-1, 1.01, 0.1), density=True, color="skyblue")
    mean_val = flattened.mean()
    
    ax.axvline(mean_val, color='black', linestyle='dashed', linewidth=1)
    ax.text(mean_val+0.3, 3, f'{mean_val:.2f}', color='black', ha='center')
    ax.set_title(cell_type)
    ax.set_ylim(0, 3.5)
    ax.set_xlim(-1, 1)
    
    ax.set_ylabel("density")
    ax.set_xlabel("Pearson r")
    ax.label_outer()

for idx in range(len(r_vals_by_cell_type.keys()), len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"6.1_corr_histograms.{format}"), dpi=300, bbox_inches="tight")

## Dendrogram clustering

#### This isn't actually being used in the paper

In [121]:
def plot_heatmap_with_clustering(distance_matrix, cell_type, plot_name, heatmap_label, communities=None):
    """This plots a heatmap for the Supplementary Information."""
    if communities is not None:
        communities_flat = [item for sublist in communities for item in sublist]
        distance_matrix = distance_matrix.loc[communities_flat, communities_flat]

    plt.figure(figsize=(len(distance_matrix)*0.1+1, len(distance_matrix)*0.1+1))
    ax = sns.heatmap(distance_matrix, cmap="coolwarm", center=0, annot=False, fmt=".2f", vmin=-1, vmax=1, cbar=True, square=True,
                cbar_kws={'label': heatmap_label, 'shrink': 0.6})
    
    # plot black boxes around the communities
    if communities is not None:
        current_pos = 0
        for community in communities:
            num_entries = len(community)
            rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries, fill=False, color='black', linewidth=1.5, clip_on=False)
            plt.gca().add_patch(rect)
            current_pos += num_entries
    
    # set font sizes to 5.5
    ax.set_xticks(np.arange(0.5, len(distance_matrix)+0.5))
    ax.set_yticks(np.arange(0.5, len(distance_matrix)+0.5))
    ax.set_xticklabels(distance_matrix.index, fontsize=5.5, rotation=90)  # Rotate for better visibility
    ax.set_yticklabels(distance_matrix.index, fontsize=5.5)
    # set the tick length to 0
    ax.tick_params(length=0)
    
    apply_greek_labels_heatmap(ax)

    plt.title(f"{cell_type}", fontsize=7)
    plt.tight_layout()

    for format in ["png", "svg"]:
        plt.savefig(f"{plot_folder}/6_{plot_name}_{cell_type}.{format}", bbox_inches="tight")
        
        
def plot_heatmap_with_clustering_small(distance_matrix, cell_type, plot_name, heatmap_label, communities=None):
    """This plots a heatmap for the Main Text."""
    if communities is not None:
        communities_flat = [item for sublist in communities for item in sublist]
        distance_matrix = distance_matrix.loc[communities_flat, communities_flat]

    plt.figure(figsize=cm_to_inches((8.5,8.5)))
    ax = sns.heatmap(
        distance_matrix, cmap="coolwarm", center=0, annot=False, fmt=".2f",
        vmin=-1, vmax=1, cbar=True, square=True,
        cbar_kws={'label': heatmap_label, 'shrink': 0.45}
    )
    cbar = ax.collections[0].colorbar
    cbar.ax.tick_params(labelsize=5.5)  # Reduce the fontsize of the colorbar labels
    
    # plot black boxes around the communities
    if communities is not None:
        current_pos = 0
        for community in communities:
            num_entries = len(community)
            rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries,
                                 fill=False, color='black', linewidth=1.5, clip_on=False)
            plt.gca().add_patch(rect)
            current_pos += num_entries
    
    # Instead of the default y-axis labels, we will stagger them in 3 positions.
    # First, remove the default labels:
    ax.set_xticklabels([])
    ax.set_yticklabels([])

    # Define 3 staggered horizontal offsets (in axis coordinates: 0=left, 1=right).
    # Adjust these values as needed for your plot.
    stagger_offsets = [-0.01, -0.15, -0.19]
    
    # Manually add the y-axis labels at staggered positions.
    for i, label in enumerate(distance_matrix.index):
        offset = stagger_offsets[i % 2]
        ax.text(
            offset, i + 0.2, introduce_greek_letters(label),
            ha='right', va='center', fontsize=5.5,
            transform=ax.get_yaxis_transform()  # x in axis fraction, y in data coordinates
        )
    
    # remove tick marks
    ax.tick_params(length=0)
    
    apply_greek_labels_heatmap(ax)
    
    plt.title(f"{cell_type}", fontsize=7)
    plt.tight_layout()

    for format in ["png", "svg"]:
        plt.savefig(f"{plot_folder}/6_small_{plot_name}_{cell_type}.{format}", bbox_inches="tight")

In [122]:
%%capture output
import scipy.cluster.hierarchy as sch
from scipy.spatial.distance import squareform

def dendrogram_clustering(distance_matrix):
    columns = distance_matrix.columns
    distance_matrix_original = distance_matrix.copy()
    
    # I. distance matrix
    # replace NaN values with 0
    distance_matrix = np.nan_to_num(distance_matrix)

    # condense the distance matrix
    condensed_distance = squareform(distance_matrix)

    # II. hierarchical clustering 
    linkage = sch.linkage(condensed_distance, method='ward')

    # get the dendrogram and ordered indices
    dendro = sch.dendrogram(linkage, labels=columns, no_plot=True)
    ordered_indices = dendro['leaves']

    # compute cophenetic distances between leaves
    _, cophenetic_distances = sch.cophenet(linkage, condensed_distance)
    # convert to a square matrix
    cophenetic_matrix = squareform(cophenetic_distances)
    cophenetic_df = pd.DataFrame(cophenetic_matrix, index=columns, columns=columns)
    cophenetic_df = cophenetic_df.iloc[ordered_indices, ordered_indices]
    cophenetic_df = cophenetic_df/cophenetic_df.max().max()
    
    return distance_matrix_original.iloc[ordered_indices, ordered_indices], cophenetic_df, linkage

for cell_type in r_vals_by_cell_type.keys():
    cytokine_filter = strong_impact_by_cell_type[cell_type][strong_impact_by_cell_type[cell_type]].index
    distance_matrix = cosine_distances_by_cell_type[cell_type].loc[cytokine_filter, cytokine_filter].astype(float)
    # distance_matrix = 1 - r_vals_by_cytokine
    ordered_distance_vals, _, _ = dendrogram_clustering(distance_matrix)
    plot_heatmap_with_clustering(1-ordered_distance_vals, cell_type, plot_name="cosine_dendrogram", heatmap_label="cosine similarity") #, family_dict=cytokine_name_to_level_2_family)

## Group cytokines by highly similar correlation values

In [123]:
# get the correlation values in long format
all_r_vals_long = (
    r_vals_by_cell_type_df
    .stack()
    .reset_index()
    .rename(columns={
        "level_2": "cell_type",
        0: "r_val"
    })
)

In [124]:
# check if both cytokines have a strong impact in a given cell type
all_r_vals_long["strong_impact"] = False
for index, row in all_r_vals_long.iterrows():
    cytokine_1 = row["cytokine1"]
    cytokine_2 = row["cytokine2"]
    cell_type = row["cell_type"]
    strong_impact_1 = strong_impact_by_cell_type.loc[cytokine_1, cell_type]
    strong_impact_2 = strong_impact_by_cell_type.loc[cytokine_2, cell_type]
    if strong_impact_1 and strong_impact_2:
        all_r_vals_long.loc[index, "strong_impact"] = True

In [125]:
# subset to strong impact and exclude unwanted cell types
all_r_vals_long = all_r_vals_long[all_r_vals_long["strong_impact"]]
all_r_vals_long = all_r_vals_long[all_r_vals_long["cell_type"].isin(ALL_CELL_TYPES_EXCLUDE)]

In [126]:
# Create a unique identifier
all_r_vals_long["unique_identifier"] = all_r_vals_long.apply(
    lambda x: "_".join(sorted([str(x.get("cytokine1", "")), str(x.get("cytokine2", "")), str(x.get("cell_type", ""))])), 
    axis=1
)
# Drop duplicates based on the unique identifier
all_r_vals_long = all_r_vals_long.drop_duplicates(subset=["unique_identifier"])

In [127]:
# subset to those that have values in at least three cell types
all_r_vals_long["cyto1_cyto2"] = all_r_vals_long["cytokine1"] + "_" + all_r_vals_long["cytokine2"]
val_counts = all_r_vals_long["cyto1_cyto2"].value_counts()

# subset to those measured in at least three cell types
val_counts = val_counts[val_counts > 2]
all_r_vals_long = all_r_vals_long[all_r_vals_long["cyto1_cyto2"].isin(val_counts.index)]

In [128]:
# calculate the mean value
mean_r = all_r_vals_long.groupby(["cytokine1", "cytokine2"])["r_val"].mean()
mean_r = mean_r.unstack()

In [129]:
# threshold at mean values
group_by_cytokine = pd.DataFrame(columns=["cytokine1", "cytokine2"])
threshold = 0.7
for cytokine_1 in mean_r.index:
    cytokine_r = mean_r.loc[cytokine_1]
    cytokine_r = cytokine_r[cytokine_r > threshold]
    for cytokine_2 in cytokine_r.index:
        group_by_cytokine.loc[len(group_by_cytokine)] = [cytokine_1, cytokine_2]

group_by_cytokine = group_by_cytokine[group_by_cytokine["cytokine1"] != group_by_cytokine["cytokine2"]]

In [130]:
# Create pair column by sorting cytokine names alphabetically and joining with underscore
group_by_cytokine["pair"] = group_by_cytokine.apply(lambda x: f"{min(x['cytokine1'], x['cytokine2'])}_{max(x['cytokine1'], x['cytokine2'])}", axis=1)

# Remove duplicate pairs
group_by_cytokine = group_by_cytokine.drop_duplicates(subset=["pair"])

In [131]:
def sort_list_by_reference(input_list, reference):
    """
    Sort a list based on the order of elements in a reference list.
    
    Parameters:
    -----------
    input_list : list
        The list to be sorted. Should be a subset of the reference list.
    reference : list
        The reference list that defines the desired order.
        
    Returns:
    --------
    list
        A new list with elements from input_list sorted according to their order in reference.
    """
    # Create a dictionary mapping each element in reference to its position
    reference_positions = {element: i for i, element in enumerate(reference)}
    
    # Sort the input list based on the positions in the reference list
    sorted_list = sorted(input_list, key=lambda x: reference_positions.get(x, float('inf')))
    
    return sorted_list

In [ ]:
import igraph as ig
import leidenalg

# Create an undirected graph
G = ig.Graph()

# Add vertices (unique cytokines)
unique_cytokines = set(group_by_cytokine['cytokine1'].unique()) | set(group_by_cytokine['cytokine2'].unique())
G.add_vertices(list(unique_cytokines))

# Add edges from the dataframe
edges = list(zip(group_by_cytokine['cytokine1'], group_by_cytokine['cytokine2']))
G.add_edges(edges)

# Find connected components
components = G.components()

# Print each group
all_close_groups = []
for i, component in enumerate(components, 1):
    # Get the vertex names for this component
    group = [G.vs[idx]['name'] for idx in component]
    sorted_group = sort_list_by_reference(group, list(cytokine_info.index))
    print(f"Group {i}: {sorted_group}")
    all_close_groups.append(sorted_group)

In [133]:
with open(os.path.join("output", "6_highly_similar_groups.pkl"), "wb") as f:
    pickle.dump(all_close_groups, f)

## Leiden clustering


#### Clustering in each cell type

In [134]:
def leiden_clustering(distance_matrix, resolution_parameter=0.5, print_communities=False, invert_distance=False):
    # Convert the DataFrame to a list-of-lists
    if invert_distance:
        adjacency = (1-distance_matrix.values).tolist()
    else:
        adjacency = (distance_matrix.values).tolist()

    # Create an undirected graph. Use loops=False to avoid self-loops.
    g = ig.Graph.Weighted_Adjacency(adjacency, mode=ig.ADJ_UNDIRECTED, attr="weight", loops=False)

    # Set vertex names for easier interpretation.
    g.vs["name"] = list(distance_matrix.index)

    # Apply the Leiden algorithm.
    partition = leidenalg.find_partition(
        g,
        leidenalg.CPMVertexPartition,
        weights='weight',
        resolution_parameter=resolution_parameter
    )

    all_communities = []
    for i, community in enumerate(partition):
        community_nodes = [g.vs[node]["name"] for node in community]
        if print_communities:
            print(f"Community {i+1}: {community_nodes}")
        all_communities.append(community_nodes)

    communities_flat = [item for sublist in all_communities for item in sublist]
    
    return all_communities, communities_flat

In [135]:
%%capture output
all_communities_by_cell_type = {}
for cell_type in ALL_CELL_TYPES_EXCLUDE:
    # subset to cytokines with strong impact in this cell type
    cytokine_filter = strong_impact_by_cell_type[cell_type][strong_impact_by_cell_type[cell_type]].index
    distance_matrix = r_vals_by_cell_type[cell_type].loc[cytokine_filter, cytokine_filter].astype(float)
    # Get the distance matrix.
    df = distance_matrix.copy()
    # Ensure that the diagonal is zero (i.e., no self-loops)
    np.fill_diagonal(df.values, 0)

    all_communities, _ = leiden_clustering(distance_matrix, 0.5, False, False)

    all_communities_by_cell_type[cell_type] = all_communities
    plot_heatmap_with_clustering(distance_matrix, cell_type, plot_name="pearson_leiden", heatmap_label="Pearson r", communities=all_communities)
    if cell_type == "CD14 Mono":
        plot_heatmap_with_clustering_small(distance_matrix, cell_type, plot_name="pearson_leiden", heatmap_label="Pearson r", communities=all_communities)

#### Clustering across cell types

In [136]:
# create a distance matrix by normalizing clustering counts
grouped_together_total = pd.DataFrame(0, index=CYTOKINE_ORDER, columns=CYTOKINE_ORDER)

for cell_type, communities in all_communities_by_cell_type.items():
    for community in communities:
        for cytokine_1 in community:
            for cytokine_2 in community:
                grouped_together_total.loc[cytokine_1, cytokine_2] += 1
                
# discard those with 0 in the diagonal
# Keep only cytokines that have at least 2 interactions
non_zero_diag = grouped_together_total.index[grouped_together_total.values.diagonal() > 1]
grouped_together_total = grouped_together_total.loc[non_zero_diag, non_zero_diag]

# Compute the row sums
row_sums = grouped_together_total.sum(axis=1)

# Compute the inverse square roots of the row sums
eps = 1e-10
inv_sqrt = 1 / np.sqrt(row_sums + eps)

# Get the normalization factors
norm_factors = pd.Series(inv_sqrt, index=grouped_together_total.index)

# Perform the symmetric normalization
# Multiply rows by the normalization factors, then columns.
normalized_matrix = grouped_together_total.mul(norm_factors, axis=0)\
                                          .mul(norm_factors, axis=1)

# round to 5 decimal places
normalized_matrix = normalized_matrix.round(5)
cosine_distances = squareform(pdist(normalized_matrix.values, metric='cosine'))
cosine_distances = pd.DataFrame(cosine_distances, index=normalized_matrix.index, columns=normalized_matrix.index)                

In [137]:
# perform Leiden clustering
all_communities, _ = leiden_clustering(cosine_distances, resolution_parameter=0.5, print_communities=False, invert_distance=True)

In [138]:
# reorder communities internally to group highly similar cytokines
reordered_communities = []

for community in all_communities:
    community_list = list(community)
    
    # track which elements were moved to maintain order of non-subgroup elements
    moved_elements = set()
    
    # process each group that's a subset of this community in all_close_groups
    for group in all_close_groups: # this was calculated above by grouping on correlation
        if set(group).issubset(set(community)):
            # find the earliest index of any group member in the community
            earliest_idx = min(community_list.index(member) for member in group)
            
            # remove group members from their current positions
            # (if they haven't been moved by a previously)
            for member in group:
                if member not in moved_elements:
                    community_list.remove(member)
                    moved_elements.add(member)
            
            # insert group members together at the earliest possible index
            for i, member in enumerate(group):
                if member not in community_list:  # avoid duplicates (do I need this?)
                    community_list.insert(earliest_idx + i, member)
    
    reordered_communities.append(community_list)

# replace all_communities with the reordered version
all_communities = reordered_communities
communities_flat = [item for sublist in all_communities for item in sublist]

In [ ]:
# -------------------- Plotting --------------------
df_to_plot = grouped_together_total.loc[communities_flat, communities_flat]
plt.figure(figsize=cm_to_inches((12, 12)))

ax = sns.heatmap(df_to_plot, square=True, vmin=0, vmax=12, cbar=True,
             cbar_kws={'label': 'grouping count', 'shrink': 0.6})

# plot white boxes around the communities
current_pos = 0
for community in all_communities:
    num_entries = len(community)
    rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries, fill=False, color='white', linewidth=1.5, clip_on=False)
    plt.gca().add_patch(rect)
    current_pos += num_entries

# Plot additional boxes around highly similar groups
for group in all_close_groups:
    group_indices = [communities_flat.index(member) for member in group if member in communities_flat]
    if group_indices:
        min_idx = min(group_indices)
        max_idx = max(group_indices)
        rect = plt.Rectangle((min_idx, min_idx), max_idx - min_idx + 1, max_idx - min_idx + 1, fill=False, color='skyblue', ls="--", linewidth=1, clip_on=False)
        plt.gca().add_patch(rect)

# set font sizes to 5.5
ax.set_xticks(np.arange(0.5, len(df_to_plot)+0.5))
ax.set_yticks(np.arange(0.5, len(df_to_plot)+0.5))
ax.set_xticklabels(df_to_plot.index, fontsize=5.5, rotation=90)  # Rotate for better visibility
ax.set_yticklabels(df_to_plot.index, fontsize=5.5)
# set the tick length to 0
ax.tick_params(length=0)

apply_greek_labels_heatmap(ax)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(f"{plot_folder}/6_grouped_together_total.{format}", bbox_inches="tight")


# -------------------- Plotting ALT --------------------
df_to_plot = grouped_together_total.loc[communities_flat, communities_flat]
plt.figure(figsize=cm_to_inches((8.5, 8.5)))

ax = sns.heatmap(df_to_plot, square=True, vmin=0, vmax=12, cbar=True,
             cbar_kws={'label': 'grouping count', 'shrink': 0.45})

cbar = ax.collections[0].colorbar
cbar.ax.tick_params(labelsize=5.5)  # Reduce the fontsize of the colorbar labels

# plot white boxes around the communities
current_pos = 0
for community in all_communities:
    num_entries = len(community)
    rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries, fill=False, color='white', linewidth=1, clip_on=False)
    plt.gca().add_patch(rect)
    current_pos += num_entries

# Plot additional boxes around highly similar groups
for group in all_close_groups:
    group_indices = [communities_flat.index(member) for member in group if member in communities_flat]
    if group_indices:
        min_idx = min(group_indices)
        max_idx = max(group_indices)
        rect = plt.Rectangle((min_idx, min_idx), max_idx - min_idx + 1, max_idx - min_idx + 1, fill=False, color='skyblue', ls="--", linewidth=1, clip_on=False)
        plt.gca().add_patch(rect)


# Instead of the default y-axis labels, we will stagger them in 3 positions.
# First, remove the default labels:
ax.set_xticklabels([])
ax.set_yticklabels([])

# Define 3 staggered horizontal offsets (in axis coordinates: 0=left, 1=right).
# Adjust these values as needed for your plot.
stagger_offsets = [-0.01, -0.15, -0.19]

# Manually add the y-axis labels at staggered positions.
# Here, the y coordinate is the same as the tick (i + 0.5) and the x position is determined by the stagger.
for i, label in enumerate(df_to_plot.index):
    offset = stagger_offsets[i % 2]
    ax.text(
        offset, i + 0.2, introduce_greek_letters(label),
        ha='right', va='center', fontsize=5.5,
        transform=ax.get_yaxis_transform()  # x in axis fraction, y in data coordinates
    )

# remove tick marks
ax.tick_params(length=0)
    
ax.set_title("cytokine grouping across cell types", fontsize=7)
apply_greek_labels_heatmap(ax)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(f"{plot_folder}/6_grouped_together_total_alt.{format}", bbox_inches="tight")


# ------------------------------------------------------------------------
df_to_plot = normalized_matrix.loc[communities_flat, communities_flat]
plt.figure(figsize=cm_to_inches((12, 12)))
ax = sns.heatmap(df_to_plot, square=True, vmin=df_to_plot.min().min(), vmax=df_to_plot.max().max(), cbar=True, cmap="viridis_r",
             cbar_kws={'label': 'relative grouping', 'shrink': 0.6})

# plot black boxes around the communities
current_pos = 0
for community in all_communities:
    num_entries = len(community)
    rect = plt.Rectangle((current_pos, current_pos), num_entries, num_entries, fill=False, color='black', linewidth=1.5, clip_on=False)
    plt.gca().add_patch(rect)
    current_pos += num_entries

# set font sizes to 5.5
ax.set_xticks(np.arange(0.5, len(df_to_plot)+0.5))
ax.set_yticks(np.arange(0.5, len(df_to_plot)+0.5))
ax.set_xticklabels(df_to_plot.index, fontsize=5.5, rotation=90)  # Rotate for better visibility
ax.set_yticklabels(df_to_plot.index, fontsize=5.5)
# set the tick length to 0
ax.tick_params(length=0)

apply_greek_labels_heatmap(ax)

plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(f"{plot_folder}/6_grouped_together_rel.{format}", bbox_inches="tight")

In [140]:
# save the results
with open(os.path.join("output", "6_all_communities_cyto_cyto.pkl"), "wb") as f:
    pickle.dump(all_communities, f)

## 6.2 - Correlations between responses in different cell types

In [141]:
plot_folder = os.path.join(base_plot_folder, "6.2_between_cell_similarity")
os.makedirs(plot_folder, exist_ok=True)

In [142]:
# calculate novel filters that define genes present in both cell types (takes ~45s)
cell_type_combs = list(combinations_with_replacement(ALL_CELL_TYPES, 2))

common_filter_by_cell_type_comb = {}
for cell_type_1, cell_type_2 in cell_type_combs:
    # get first filter (as for within cell calculations)
    all_inclusive_filter_1 = expressed_genes_by_cell_type_dict[cell_type_1]
    existing_fold_change_vals_1 = edger_df_log2FoldChange[cell_type_1].dropna(how='all').index
    all_inclusive_filter_1 = [gene for gene in all_inclusive_filter_1 if gene in existing_fold_change_vals_1]

    # get second filter(as for within cell calculations)
    all_inclusive_filter_2 = expressed_genes_by_cell_type_dict[cell_type_2]
    existing_fold_change_vals_2 = edger_df_log2FoldChange[cell_type_2].dropna(how='all').index
    all_inclusive_filter_2 = [gene for gene in all_inclusive_filter_2 if gene in existing_fold_change_vals_2]

    # get the intersection of the two filters
    all_inclusive_filter = [gene for gene in all_inclusive_filter_1 if gene in all_inclusive_filter_2]
    common_filter_by_cell_type_comb[f"{cell_type_1}_{cell_type_2}"] = all_inclusive_filter
    
    # print results
    # print(f"Common filter for {cell_type_1} and {cell_type_2}: {len(all_inclusive_filter)} genes")

In [143]:
%%capture output
def scatter_plot(x, y, cytokine, cell1, cell2):
    """Larger scatter plot for the SI."""
    plt.figure(figsize=cm_to_inches((3.8, 3.5)))
    
    # create a cross at the origin
    plt.plot([-5, 5], [0, 0], color="black", linewidth=1, zorder=0, alpha=0.5)
    plt.plot([0, 0], [-5, 5], color="black", linewidth=1, zorder=0, alpha=0.5)
    plt.scatter(x, y, s=1, edgecolors="none", rasterized=True, zorder=1)
    r = np.corrcoef(x, y)[0, 1]
    plt.xlim(-4, 4)
    plt.ylim(-4, 4)
    plt.xticks([-4, -2, 0, 2, 4])
    plt.yticks([-4, -2, 0, 2, 4]) 
    plt.title(f"{cytokine}", fontsize=6.5)
    # add r on the top left
    plt.text(-3.5, 2.65, f"r = {r:.2f}", fontsize=6.5)
    plt.xlabel(f"log$_{2}$FC({cell1})", fontsize=6.5)
    plt.ylabel(f"log$_{2}$FC({cell2})", fontsize=6.5)
    plt.tight_layout()
    for format in ["png", "svg"]:
        plt.savefig(os.path.join(plot_folder, f"6.2_scatter_{cell1}_{cell2}_{cytokine}.{format}"), dpi=400)
    plt.close()

def scatter_plot_small(x, y, cytokine, cell1, cell2):
    """Smaller scatter plot for the Main Text."""
    plt.figure(figsize=cm_to_inches((3.2, 3)))

    plt.scatter(x, y, s=1, edgecolors="none", rasterized=True, zorder=1)
    r = np.corrcoef(x, y)[0, 1]
    plt.xlim(-4, 4)
    plt.ylim(-4, 4)
    plt.xticks([-4, 0, 4], fontsize=6)
    plt.yticks([-4, 0, 4], fontsize=6) 

    plt.xlabel(f"log$_{2}$FC", fontsize=6)
    plt.ylabel(f"log$_{2}$FC", fontsize=6)
    plt.title(f"{cell1} vs {cell2}", fontsize=6)
    
    # add r on the top left
    plt.text(-3.5, 2.5, f"r = {r:.2f}", fontsize=6)
    
    plt.tight_layout()
    for format in ["png", "svg"]:
        plt.savefig(os.path.join(plot_folder, f"6.2_scatter_small_{cell1}_{cell2}_{cytokine}.{format}"), dpi=400)
    plt.close()

pairs_to_plot = [
    # ("B cell", "CD4 T cell", "4-1BBL"),
    # ("B cell", "CD4 T cell", "APRIL"),
    # ("B cell", "CD4 T cell", "IL-4"),
    # ("B cell", "CD4 T cell", "IL-13"),
    # ("B cell", "CD4 T cell", "IL-15"),
    # ("B cell", "CD4 T cell", "TSLP"),
    # ("B cell", "CD4 T cell", "TPO"),
    # ("B cell", "CD14 Mono", "4-1BBL"),
    # ("B cell", "CD14 Mono", "APRIL"),
    # ("B cell", "CD14 Mono", "IL-4"),
    # ("B cell", "CD14 Mono", "IL-13"),
    # ("B cell", "CD14 Mono", "IL-15"),
    # ("B cell", "CD14 Mono", "TSLP"),
    # ("B cell", "CD14 Mono", "TPO"),
    ("CD14 Mono", "CD16 Mono", "GM-CSF"),
    ("CD14 Mono", "cDC", "GM-CSF"),
    ("CD14 Mono", "B cell", "GM-CSF"),
]

for cell_type_1, cell_type_2, cytokine in pairs_to_plot:
    try:
        all_inclusive_filter = common_filter_by_cell_type_comb[f"{cell_type_1}_{cell_type_2}"]
    except KeyError:
        all_inclusive_filter = common_filter_by_cell_type_comb[f"{cell_type_2}_{cell_type_1}"]
    cell_types_log2 = edger_df_log2FoldChange.loc[all_inclusive_filter, :]
    
    # get the values
    vals_1 = cell_types_log2[cell_type_1][cytokine].dropna()
    vals_2 = cell_types_log2[cell_type_2][cytokine].dropna()
    
    # use the intersection of common genes
    common_index = vals_1.index.intersection(vals_2.index)
    vals_1 = vals_1.loc[common_index]
    vals_2 = vals_2.loc[common_index]
    
    scatter_plot(vals_1, vals_2, cytokine, cell_type_1, cell_type_2)
    scatter_plot_small(vals_1, vals_2, cytokine, cell_type_1, cell_type_2)

### calculate correlation vals

In [144]:
if os.path.exists(os.path.join("output", f"6.2_r_vals_cell_cell_df.pkl")):
    with open(os.path.join("output", f"6.2_r_vals_cell_cell_df.pkl"), "rb") as f:
        r_vals_cell_cell_df = pickle.load(f)
else:
    r_vals_cell_cell = []
    for cell_type_1, cell_type_2 in cell_type_combs:
        # print(f"Calculating r_vals_cell_cell for {cell_type_1} and {cell_type_2}")
        
        # get the filter
        all_inclusive_filter = common_filter_by_cell_type_comb[f"{cell_type_1}_{cell_type_2}"]
        cell_types_log2 = edger_df_log2FoldChange.loc[all_inclusive_filter, :]

        for cytokine in CYTOKINE_ORDER:
            if cell_type_1 == cell_type_2:
                r_vals_cell_cell.append({
                    "cytokine": cytokine,
                    "cell_type_1": cell_type_1,
                    "cell_type_2": cell_type_2,
                    "r": 1.0
                })
                continue
            
            # get the values
            vals_1 = cell_types_log2[cell_type_1][cytokine].dropna()
            vals_2 = cell_types_log2[cell_type_2][cytokine].dropna()
            
            # use the intersection of common genes
            common_index = vals_1.index.intersection(vals_2.index)
            vals_1 = vals_1.loc[common_index]
            vals_2 = vals_2.loc[common_index]

            if len(vals_1) == 0 or len(vals_2) == 0:
                r_vals_cell_cell.append({
                    "cytokine": cytokine,
                    "cell_type_1": cell_type_1,
                    "cell_type_2": cell_type_2,
                    "r": np.nan
                })
                continue

            r_vals_cell_cell.append({
                "cytokine": cytokine,
                "cell_type_1": cell_type_1,
                "cell_type_2": cell_type_2,
                "r": np.corrcoef(vals_1, vals_2)[0, 1]
                })
            
    r_vals_cell_cell_df = pd.DataFrame(r_vals_cell_cell)
    r_vals_cell_cell_df = r_vals_cell_cell_df.pivot(index="cell_type_1", columns=["cytokine", "cell_type_2"], values="r")
    
    # symmetrize values
    filled_dfs = {}
    for cytokine in r_vals_cell_cell_df.columns.levels[0]:
        # Extract the matrix corresponding to the current cytokine.
        matrix = r_vals_cell_cell_df.xs(cytokine, axis=1, level=0)
        
        # Fill in the missing values using the symmetric (transposed) value.
        matrix_filled = matrix.combine_first(matrix.T)
        
        # Save the filled matrix to a dictionary
        filled_dfs[cytokine] = matrix_filled

    # Concatenate the filled matrices back into a single DataFrame
    r_vals_cell_cell_df = pd.concat(filled_dfs, axis=1)

    # rename columns and the index
    r_vals_cell_cell_df.columns.names = ['cytokine', 'cell_type_2']
    r_vals_cell_cell_df.index.name = "cell_type_1"
    
    # order by ALL_CELL_TYPES
    r_vals_cell_cell_df = r_vals_cell_cell_df.reindex(index=ALL_CELL_TYPES, columns=pd.MultiIndex.from_product([r_vals_cell_cell_df.columns.levels[0], ALL_CELL_TYPES]))
    
    with open(os.path.join("output", f"6.2_r_vals_cell_cell_df.pkl"), "wb") as f:
        pickle.dump(r_vals_cell_cell_df, f)

### make various scatter plots and heatmaps

In [ ]:
reference_cell_type = "CD14 Mono"
other_cell_types_to_plot = ["B cell", "CD4 T cell", "NK CD56low", "cDC", "CD16 Mono", "HSPC"]
cytokines = ["GM-CSF", "IL-3", "IL-4", "4-1BBL", "APRIL"]

triplexes = [(reference_cell_type, other_cell_type, cytokine) 
             for cytokine in cytokines
             for other_cell_type in other_cell_types_to_plot
             ]

# ------------------------- Large plot -------------------------
fig, axs = plt.subplots(nrows=5, ncols=6, figsize=cm_to_inches((16, 12)), sharex=True, sharey=True)
axs = axs.flatten()
counter = 0

for cell_type_1, cell_type_2, cytokine in triplexes:
    try:
        all_inclusive_filter = common_filter_by_cell_type_comb[f"{cell_type_1}_{cell_type_2}"]
    except KeyError:
        all_inclusive_filter = common_filter_by_cell_type_comb[f"{cell_type_2}_{cell_type_1}"]
    cell_types_log2 = edger_df_log2FoldChange.loc[all_inclusive_filter, :]
        
    # get the values
    vals_1 = cell_types_log2[cell_type_1][cytokine].dropna()
    vals_2 = cell_types_log2[cell_type_2][cytokine].dropna()
    
    # use the intersection
    common_index = vals_1.index.intersection(vals_2.index)
    vals_1 = vals_1.loc[common_index]
    vals_2 = vals_2.loc[common_index]
    
    r = np.corrcoef(vals_1, vals_2)[0, 1]
    
    # ---- subplots ----
    ax = axs[counter]
    
    # create a cross at the origin
    ax.plot([-5, 5], [0, 0], color="black", linewidth=1, zorder=0, alpha=0.5)
    ax.plot([0, 0], [-5, 5], color="black", linewidth=1, zorder=0, alpha=0.5)
    ax.scatter(vals_1, vals_2, s = 1, edgecolor="none", alpha=0.5, rasterized=True, zorder=1)
    
    ax.set_xlim(-4, 4)
    ax.set_ylim(-4, 4)
    ax.set_xticks([-4, -2, 0, 2, 4])
    ax.set_yticks([-4, -2, 0, 2, 4]) 

    ax.set_xlabel(f"{cell_type_1}", fontsize=6)
    ax.set_ylabel(f"{cell_type_2}", fontsize=6)
    # ax.set_xlabel(f"{introduce_greek_letters(cytokine1)}", fontsize=6)
    # ax.set_ylabel(f"{introduce_greek_letters(cytokine2)}", fontsize=6)
    #ax.set_title(f'{cytokine}, r={r:.2f}', fontsize=6)
    ax.text(0.05, 0.95, f"r={r:.2f}", transform=ax.transAxes, 
            fontsize=6, verticalalignment='top',
            bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.7, edgecolor='none'))
    
    # Set tick length to 0
    ax.tick_params(axis='both', which='both', length=0)
    
    counter += 1
    if not ax.get_subplotspec().is_first_col():
        plt.setp(ax.get_yticklabels(), visible=False)
    
    if not ax.get_subplotspec().is_last_row():
        plt.setp(ax.get_xticklabels(), visible=False)

plt.tight_layout(pad=0.5, h_pad=0.5, w_pad=1)
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"6.2_corr_panel.{format}"),
                    bbox_inches="tight", dpi=400)

In [146]:
# remove unwanted cell types
cell_types_to_exclude = ["HSPC", "pDC", "ILC"]
r_vals_cell_cell_df_filter = r_vals_cell_cell_df[~r_vals_cell_cell_df.index.isin(cell_types_to_exclude)]
r_vals_cell_cell_df_filter = r_vals_cell_cell_df_filter.loc[:, (slice(None), ~r_vals_cell_cell_df_filter.columns.get_level_values(1).isin(cell_types_to_exclude))]

In [ ]:
%%capture output
for cytokine in CYTOKINE_ORDER:
    if cytokine not in ["GM-CSF", "IL-4", "TSLP"]:
        continue
    plt.figure(figsize=cm_to_inches((6, 5)))
    sns.heatmap(r_vals_cell_cell_df_filter.xs(cytokine, axis=1, level=0), cmap="rocket", vmin=0, vmax=1,
                cbar_kws={"label": "Pearson r"})
    plt.tick_params(length=0)
    plt.title(f"{cytokine}", fontsize=7)
    plt.tight_layout()
    for format in ["png", "svg"]:
        plt.savefig(os.path.join(plot_folder, f"6.2_cell_cell_similarity_{cytokine}.{format}"), dpi=300)
    plt.close()

In [ ]:
cytokines_to_plot = [
    "4-1BBL", "APRIL", "IL-2", "IL-15",
    "IL-4", "IL-13", "IL-21", "CD40L",
    "M-CSF", "GM-CSF", "IL-3", "C5a",
    "TSLP", "IL-32-beta", "IFN-omega", "IFN-gamma",
]

# Determine grid dimensions; here we force a 4×4 grid.
n_plots = len(cytokines_to_plot)
n_cols = 4
n_rows = int(np.ceil(n_plots / n_cols))

# Create a figure with subplots.
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=cm_to_inches((18, 16)))
# Flatten the axes array for easy iteration.
axes = axes.flatten()

# Loop over each cytokine and plot its heatmap in the corresponding subplot.
for i, cytokine in enumerate(cytokines_to_plot):
    ax = axes[i]
    
    # Extract the data for this cytokine.
    data = r_vals_cell_cell_df_filter.xs(cytokine, axis=1, level=0)
    
    # Plot the heatmap without its own colorbar.
    sns.heatmap(data,
                cmap="rocket",
                vmin=0, vmax=1,
                cbar=False,
                ax=ax,
                square=True)
    
    # Give each subplot its own title.
    ax.set_title(f"{cytokine}", fontsize=7)
    
    # Remove tick labels from interior subplots:
    # - Only show x tick labels on the bottom row.
    # - Only show y tick labels on the left column.
    row = i // n_cols
    col = i % n_cols
    if row != n_rows - 1:  # not bottom row
        ax.set_xticklabels([])
    if col != 0:          # not left column
        ax.set_yticklabels([])
    
    ax.set_ylabel("")
    # Optionally remove tick marks.
    ax.tick_params(length=0)

# Remove any unused (empty) subplots (if the total number of cytokines is less than n_rows*n_cols)
for j in range(n_plots, len(axes)):
    fig.delaxes(axes[j])

# Add one shared colorbar on the right side.
# First, create a ScalarMappable with the same normalization and colormap as used in the heatmaps.
norm = plt.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap="rocket", norm=norm)
sm.set_array([])

# Adjust the figure layout to make room for the colorbar.
fig.subplots_adjust(right=0.85, wspace=0.3, hspace=0.3)
cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.35])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Pearson r")

# apply tight_layout
plt.tight_layout()

# Save the complete grid as PNG and SVG.
for fmt in ["png", "svg"]:
    fig.savefig(os.path.join(plot_folder, f"6.2_cell_cell_similarity_grid.{fmt}"), dpi=300)

plt.show()

In [ ]:
curr_df = r_vals_cell_cell_df_filter.T.groupby(level=1).mean()
curr_df = curr_df[curr_df.index]
plt.figure(figsize=cm_to_inches((6, 5)))
sns.heatmap(curr_df, cmap="rocket", vmin=0, vmax=1,
            cbar_kws={"label": "Pearson r"}, square=True)
plt.tick_params(length=0)
plt.xlabel("")
plt.ylabel("")
plt.title(f"mean response similarity", fontsize=7)
plt.tight_layout()
for format in ["png", "svg"]:
    plt.savefig(os.path.join(plot_folder, f"6.2_mean_cell_cell_similarity.{format}"), dpi=300)

### calculate and plot mean cell-cell correlation values

In [150]:
# set all diagonals to NaN
for cell_type in r_vals_cell_cell_df_filter.index:
    r_vals_cell_cell_df_filter.loc[cell_type, (slice(None), cell_type)] = np.nan

def get_results_from_eval_subset(df, eval_subset):
    df = df.loc[eval_subset, (slice(None), eval_subset)]
    
    df.index.name = "cell_type_1"
    df.columns.names = ["cytokine", "cell_type_2"]

    # flatten mean_cell_cell_r_vals
    df = df.melt(ignore_index=False).dropna()
    mean_cell_cell_r_vals = df.groupby("cytokine")["value"].mean()
    std_cell_cell_r_vals = df.groupby("cytokine")["value"].std()
    
    return mean_cell_cell_r_vals, std_cell_cell_r_vals
    
cell_eval_subset = ["B cell", "CD4 T cell", "NK CD56low", "cDC", "CD14 Mono"]
mean_cell_cell_r_vals, std_cell_cell_r_vals =  get_results_from_eval_subset(r_vals_cell_cell_df_filter.copy(), cell_eval_subset)

other_subsets = [
    ["B cell", "CD4 T cell"],
    ["B cell", "NK CD56low"],
    ["B cell", "cDC"],
    ["B cell", "CD14 Mono"],
]
mean_by_subset = {}
for subset in other_subsets:
    mean_by_subset[f"{subset[0]}-{subset[1]}"], _ =  get_results_from_eval_subset(r_vals_cell_cell_df_filter.copy(), subset)

In [ ]:
# histogram of the overall mean correlations
plt.figure(figsize=cm_to_inches((4,3)))
plt.hist(mean_cell_cell_r_vals, color="skyblue", edgecolor="black", bins=np.arange(0.3,1.05,0.025))
plt.xlabel("mean Pearson r")
plt.ylabel("count")
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "6.2_mean_correlation_vals.svg"))

In [ ]:
# create a line plot of mean_cell_cell_r_vals with the cytokine names as the x-axis, ordered from low to high mean pearson r
# this serves as an overview figure of the distribution of mean correlation values

plt.figure(figsize=cm_to_inches((18,2.7)))
sorted_vals = mean_cell_cell_r_vals.sort_values()
plt.plot(range(len(sorted_vals)), sorted_vals, '-', markersize=2, color="black", label="mean")

# Plot standard deviation as shaded area
plt.fill_between(range(len(sorted_vals)), 
                 sorted_vals - std_cell_cell_r_vals[sorted_vals.index],
                 sorted_vals + std_cell_cell_r_vals[sorted_vals.index],
                 alpha=0.2, color='gray')

# plot the various subset to show how it varies by cell similarity
plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-cDC"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["cDC"], alpha=0.5, label="B cell-cDC")
plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-CD14 Mono"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["CD14 Mono"], alpha=0.5, label="B cell-CD14 Mono")
plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-CD4 T cell"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["CD4 T cell"], alpha=0.5, label="B cell-CD4 T cell")
plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-NK CD56low"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["NK CD56low"], alpha=0.5, label="B cell-NK CD56low")

# Find the index where mean goes above 0.6 and draw a vertical line there
threshold = 0.6
above_threshold_indices = np.where(sorted_vals.values > threshold)[0]
if len(above_threshold_indices) > 0:
    first_above_threshold_idx = above_threshold_indices[0]
    # Add a vertical dashed line at the threshold point
    plt.axvline(x=first_above_threshold_idx+0.5, color='red', linestyle='--', linewidth=1)
    # Add text annotation
    plt.text(first_above_threshold_idx + 1, 0.25, f'r < {threshold}', 
             color='red', fontsize=7, verticalalignment='bottom')

plt.legend(ncol=5)
plt.xticks(range(len(sorted_vals)), sorted_vals.index, rotation=90, ha='center', fontsize=5.5)
plt.ylabel('mean cell-cell\nPearson r')
plt.ylim(0.25, 0.9)
apply_greek_labels_heatmap(plt.gca())
plt.margins(x=0.01) # Remove whitespace on left and right
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "6.2_mean_correlation_vals_ordered.svg"))

In [ ]:
top10_least_similar_pearson_corr = sorted_vals.head(10).index
top10_least_similar_pearson_corr

### how do the mean correlation values depend on the distributon of response magnitudes?

In [153]:
# get the mean and std of magnitudes (for the eval subset)
normalized_distances_6_2_mean = all_normalized_distances_mean[cell_eval_subset].mean(axis=1)
normalized_distances_6_2_std = all_normalized_distances_mean[cell_eval_subset].std(axis=1)

# get the mean and std of the relative number of DEGs (for the eval subset)
# I don't think I actually use this for anything here (?)
significant_results_rel = significant_results.loc[cell_eval_subset]
significant_results_rel = significant_results_rel.div(significant_results_rel.max(axis=1), axis=0)
significant_results_rel_6_2_mean = significant_results_rel.mean(axis=0)
significant_results_rel_6_2_std = significant_results_rel.std(axis=0)

In [ ]:
comparison_df_6_2 = pd.DataFrame({
    'mean_cell_cell_r': mean_cell_cell_r_vals,
    'normalized_distance_std': normalized_distances_6_2_std
})

plt.figure(figsize=cm_to_inches((3.6,3.6)))
r = np.corrcoef(comparison_df_6_2['normalized_distance_std'], comparison_df_6_2['mean_cell_cell_r'])[0,1]
plt.scatter(comparison_df_6_2['normalized_distance_std'], comparison_df_6_2['mean_cell_cell_r'], s=2)
plt.xlabel(r"$\sigma$(response magnitude)")
plt.ylabel("mean cell-cell\nPearson r")
plt.title(f"r = {r:.2f}", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "6.2_mean_correlation_vals_std_distance.svg"))

# Look at individual top genes

In [ ]:
def get_top_n_genes_for_single_cytokine(cell_type, cytokine, n, by="fold_change", direction="from_top", filter_protein_coding=False):
    # get significant genes
    p_vals = edger_df_padj[(cell_type, cytokine)]
    significance = (p_vals < 0.05)
    if filter_protein_coding:
        significance = significance[significance.index.isin(protein_coding_genes)]
    significance = significance[significance]
    
    fold_changes = edger_df_log2FoldChange.loc[significance.index, (cell_type, cytokine)]
    fold_changes = fold_changes.sort_values(ascending=False)
    p_vals = p_vals.sort_values(ascending=True)
    
    if by=="fold_change":
        if direction=="from_top":
            fold_changes = fold_changes.iloc[:n]
        elif direction=="from_bottom":
            fold_changes = fold_changes.iloc[-n:]
        else:
            raise Exception("direction must be from_top or from_bottom")
        p_vals = p_vals[fold_changes.index]
    elif by=="pval":
        p_vals = p_vals[:n]
        fold_changes = fold_changes[significance.index]
    
    return_df = pd.DataFrame({
        "pval": p_vals,
        "FC": fold_changes
    })
    
    return return_df

def get_top_values_both_directions(cell_type, cytokine, n, filter_protein_coding=True):
    # get dfs from top and bottom
    df_top = get_top_n_genes_for_single_cytokine(cell_type, cytokine, n=n, direction="from_top", filter_protein_coding=filter_protein_coding)
    df_bottom = get_top_n_genes_for_single_cytokine(cell_type, cytokine, n=n, direction="from_bottom", filter_protein_coding=filter_protein_coding)

    # merge them
    df = pd.concat([df_top, df_bottom])
    df = df[~df.index.duplicated(keep='first')]

    # generate a unique identifier
    df["direction"] = df["FC"].apply(lambda x: "upregulation" if x > 0 else "downregulation")
    df = df.reset_index().rename(columns={'index': 'gene'})
    df["identifier"] =  df["gene"] + "," + df["direction"]

    # get the top n entries
    df["abs_FC"] = abs(df["FC"])
    df = df.sort_values(by="abs_FC", ascending=False)
    df = df[:n]
    
    return df

def jaccard_index(set1, set2):
    intersection = set1.intersection(set2)
    union = set1.union(set2)
    
    return len(intersection)/len(union)

In [213]:
# this runs for 5 min
n = 100
intersection_vals_cell_cell = []
for cytokine in CYTOKINE_ORDER:
    for cell_type_1, cell_type_2 in cell_type_combs:
        df_cell_1 = get_top_values_both_directions(cell_type_1, cytokine, n=n, filter_protein_coding=True)
        df_cell_2 = get_top_values_both_directions(cell_type_2, cytokine, n=n, filter_protein_coding=True)

        intersection = set(df_cell_1["identifier"].to_list()).intersection(set(df_cell_2["identifier"].to_list()))
        row_1 = {
            "cell_type_1": cell_type_1,
            "cell_type_2": cell_type_2,
            "cytokine": cytokine,
            "frac_intersection": len(intersection)/n
        }
        
        row_2 = {
            "cell_type_1": cell_type_2,
            "cell_type_2": cell_type_1,
            "cytokine": cytokine,
            "frac_intersection": len(intersection)/n
        }
        
        intersection_vals_cell_cell.append(row_1)
        if cell_type_1 != cell_type_2:
            intersection_vals_cell_cell.append(row_2)
intersection_vals_cell_cell_df = pd.DataFrame(intersection_vals_cell_cell)

intersection_vals_cell_cell_df = intersection_vals_cell_cell_df.pivot(index = "cell_type_1", columns = ["cytokine", "cell_type_2"], values = "frac_intersection")
intersection_vals_cell_cell_df = intersection_vals_cell_cell_df.loc[ALL_CELL_TYPES, (slice(None), ALL_CELL_TYPES)]

In [228]:
intersection_df_filter = intersection_vals_cell_cell_df[~intersection_vals_cell_cell_df.index.isin(cell_types_to_exclude)]
intersection_df_filter = intersection_df_filter.loc[:, (slice(None), ~intersection_df_filter.columns.get_level_values(1).isin(cell_types_to_exclude))]

In [ ]:
cytokines_to_plot = [
    "4-1BBL", "APRIL", "IL-2", "IL-15",
    "IL-4", "IL-13", "IL-21", "CD40L",
    "M-CSF", "GM-CSF", "IL-3", "C5a",
    "TSLP", "IL-32-beta", "IFN-omega", "IFN-gamma",
]

# Determine grid dimensions; here we force a 4×4 grid.
n_plots = len(cytokines_to_plot)
n_cols = 4
n_rows = int(np.ceil(n_plots / n_cols))

# Create a figure with subplots.
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=cm_to_inches((18, 16)))
# Flatten the axes array for easy iteration.
axes = axes.flatten()

# Loop over each cytokine and plot its heatmap in the corresponding subplot.
for i, cytokine in enumerate(cytokines_to_plot):
    ax = axes[i]
    
    # Extract the data for this cytokine.
    data = intersection_df_filter.xs(cytokine, axis=1, level=0)
    
    # Plot the heatmap without its own colorbar.
    sns.heatmap(data,
                cmap="rocket",
                vmin=0, vmax=1,
                cbar=False,
                ax=ax,
                square=True)
    
    # Give each subplot its own title.
    ax.set_title(f"{cytokine}", fontsize=7)
    
    # Remove tick labels from interior subplots:
    # - Only show x tick labels on the bottom row.
    # - Only show y tick labels on the left column.
    row = i // n_cols
    col = i % n_cols
    if row != n_rows - 1:  # not bottom row
        ax.set_xticklabels([])
    if col != 0:          # not left column
        ax.set_yticklabels([])
    
    ax.set_xlabel("")
    ax.set_ylabel("")
    # Optionally remove tick marks.
    ax.tick_params(length=0)

# Remove any unused (empty) subplots (if the total number of cytokines is less than n_rows*n_cols)
for j in range(n_plots, len(axes)):
    fig.delaxes(axes[j])

# Add one shared colorbar on the right side.
# First, create a ScalarMappable with the same normalization and colormap as used in the heatmaps.
norm = plt.Normalize(vmin=0, vmax=1)
sm = plt.cm.ScalarMappable(cmap="rocket", norm=norm)
sm.set_array([])

# Adjust the figure layout to make room for the colorbar.
fig.subplots_adjust(right=0.85, wspace=0.3, hspace=0.3)
cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.35])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Fraction shared top genes")

# apply tight_layout
plt.tight_layout()

# Save the complete grid as PNG and SVG.
for fmt in ["png", "svg"]:
    fig.savefig(os.path.join(plot_folder, f"6.3_ALT_cell_cell_similarity_grid.{fmt}"), dpi=300)

plt.show()

In [233]:
def get_results_from_eval_subset(df, eval_subset):
    df = df.loc[eval_subset, (slice(None), eval_subset)]
    
    df.index.name = "cell_type_1"
    df.columns.names = ["cytokine", "cell_type_2"]

    # flatten mean_cell_cell_r_vals
    df = df.melt(ignore_index=False).dropna()
    mean_cell_cell_r_vals = df.groupby("cytokine")["value"].mean()
    std_cell_cell_r_vals = df.groupby("cytokine")["value"].std()
    
    return mean_cell_cell_r_vals, std_cell_cell_r_vals
    
cell_eval_subset = ["B cell", "CD4 T cell", "NK CD56low", "cDC", "CD14 Mono"]
mean_cell_cell_inter, std_cell_cell_inter =  get_results_from_eval_subset(intersection_df_filter.copy(), cell_eval_subset)

In [ ]:
# create a line plot of mean_cell_cell_r_vals with the cytokine names as the x-axis, ordered from low to high mean pearson r
# this serves as an overview figure of the distribution of mean correlation values

plt.figure(figsize=cm_to_inches((18,2.7)))
sorted_vals = mean_cell_cell_inter.sort_values()
plt.plot(range(len(sorted_vals)), sorted_vals, '-', markersize=2, color="black", label="mean")

# Plot standard deviation as shaded area
plt.fill_between(range(len(sorted_vals)), 
                 sorted_vals - std_cell_cell_r_vals[sorted_vals.index],
                 sorted_vals + std_cell_cell_r_vals[sorted_vals.index],
                 alpha=0.2, color='gray')

# # plot the various subset to show how it varies by cell similarity
# plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-cDC"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["cDC"], alpha=0.5, label="B cell-cDC")
# plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-CD14 Mono"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["CD14 Mono"], alpha=0.5, label="B cell-CD14 Mono")
# plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-CD4 T cell"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["CD4 T cell"], alpha=0.5, label="B cell-CD4 T cell")
# plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-NK CD56low"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["NK CD56low"], alpha=0.5, label="B cell-NK CD56low")

# # Find the index where mean goes above 0.6 and draw a vertical line there
# threshold = 0.6
# above_threshold_indices = np.where(sorted_vals.values > threshold)[0]
# if len(above_threshold_indices) > 0:
#     first_above_threshold_idx = above_threshold_indices[0]
#     # Add a vertical dashed line at the threshold point
#     plt.axvline(x=first_above_threshold_idx+0.5, color='red', linestyle='--', linewidth=1)
#     # Add text annotation
#     plt.text(first_above_threshold_idx + 1, 0.25, f'r < {threshold}', 
#              color='red', fontsize=7, verticalalignment='bottom')

# plt.legend(ncol=5)
plt.xticks(range(len(sorted_vals)), sorted_vals.index, rotation=90, ha='center', fontsize=5.5)
plt.ylabel('mean cell-cell\nfraction shared top genes')
plt.ylim(0.1, 0.7)
apply_greek_labels_heatmap(plt.gca())
plt.margins(x=0.01) # Remove whitespace on left and right
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "6.3_ALT_mean_similarity_vals_ordered.svg"))

In [245]:
top10_least_similar_shared_genes = sorted_vals.head(10).index
top10_least_similar_shared_genes

In [ ]:
top10_least_similar_shared_genes.intersection(top10_least_similar_pearson_corr)

In [253]:
top10_least_similar_df = pd.DataFrame({
    "pearson": {index: gene for index, gene in enumerate(top10_least_similar_pearson_corr)},
    "shared_genes": {index: gene for index, gene in enumerate(top10_least_similar_shared_genes)},
})

In [ ]:
# Calculate Spearman rank correlation between the two ranking methods
from scipy.stats import spearmanr

# Convert to lists for spearmanr calculation
pearson_list = [top10_least_similar_df['pearson'][i] for i in range(10)]
shared_genes_list = [top10_least_similar_df['shared_genes'][i] for i in range(10)]

# Create a mapping of cytokines to their positions in each list
pearson_positions = {cytokine: i for i, cytokine in enumerate(pearson_list)}
shared_genes_positions = {cytokine: i for i, cytokine in enumerate(shared_genes_list)}

# Get common cytokines between both lists
common_cytokines = set(pearson_list).intersection(set(shared_genes_list))

# Extract positions for common cytokines
pearson_ranks = [pearson_positions[cytokine] for cytokine in common_cytokines]
shared_genes_ranks = [shared_genes_positions[cytokine] for cytokine in common_cytokines]

# Calculate Spearman correlation
if len(common_cytokines) > 1:
    correlation, p_value = spearmanr(pearson_ranks, shared_genes_ranks)
    print(f"Spearman correlation: {correlation:.2f}, p-value: {p_value:.4f}")
else:
    print("Not enough common cytokines to calculate correlation")

In [ ]:
comparison_df_6_3 = pd.DataFrame({
    'mean_cell_cell_r': mean_cell_cell_inter,
    'normalized_distance_std': normalized_distances_6_2_std
})

plt.figure(figsize=cm_to_inches((3.6,3.6)))
r = np.corrcoef(comparison_df_6_3['normalized_distance_std'], comparison_df_6_3['mean_cell_cell_r'])[0,1]
plt.scatter(comparison_df_6_3['normalized_distance_std'], comparison_df_6_3['mean_cell_cell_r'], s=2)
plt.xlabel(r"$\sigma$(response magnitude)")
plt.ylabel("mean cell-cell\ntop gene fraction shared")
plt.title(f"r = {r:.2f}", fontsize=7)
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "6.3_mean_shared_genes_std_distance.svg"))

# CODE DEVELOPMENT

In [ ]:
raise Exception("In Development - Stopping auto execution")

### Alternative similarity measures for cell-cell similarity

In [ ]:
cell_types_exclude_wo_HSPC = [cell_type for cell_type in ALL_CELL_TYPES_EXCLUDE if not cell_type=="HSPC"]
reference_cell_types = cell_types_exclude_wo_HSPC
other_cell_types_to_plot = cell_types_exclude_wo_HSPC
cytokines = CYTOKINE_ORDER

triplexes = [(reference_cell_type, other_cell_type, cytokine) 
             for cytokine in cytokines
             for reference_cell_type in reference_cell_types
             for other_cell_type in other_cell_types_to_plot
             ]

cell_types_log2_fill = edger_df_log2FoldChange.fillna(0)
all_similarities = []
total = 0
for cell_type_1, cell_type_2, cytokine in triplexes:
    if total % 100 == 0:
        print(f"Processed {total} of {len(triplexes)}")
    try:
        all_inclusive_filter = common_filter_by_cell_type_comb[f"{cell_type_1}_{cell_type_2}"]
    except KeyError:
        all_inclusive_filter = common_filter_by_cell_type_comb[f"{cell_type_2}_{cell_type_1}"]
    vals_1 = cell_types_log2[cell_type_1][cytokine].dropna()
    vals_2 = cell_types_log2[cell_type_2][cytokine].dropna()
    
    # vals_1 = cell_types_log2[cell_type_1][cytokine].dropna()
    # vals_2 = cell_types_log2[cell_type_2][cytokine].dropna()
    # common_index = vals_1.index.intersection(vals_2.index)
    # vals_1 = vals_1.loc[common_index]
    # vals_2 = vals_2.loc[common_index]
    vals_1 = cell_types_log2[cell_type_1][cytokine]
    vals_2 = cell_types_log2[cell_type_2][cytokine]
    
    similarity = (vals_1*vals_2).sum() / len(vals_1)
    
    similarity = {
        "cell_type_1": cell_type_1,
        "cell_type_2": cell_type_2,
        "cytokine": cytokine,
        "similarity": similarity,
    }
    
    all_similarities.append(similarity)
    total = total + 1
    # if cell_type_1 != cell_type_2:
    #     break
    
all_similarities = pd.DataFrame.from_dict(all_similarities)

In [633]:
all_similarities_pivot = all_similarities.pivot(index="cell_type_1", columns=["cytokine", "cell_type_2"], values="similarity")

In [636]:
all_similarities_pivot = np.log10(all_similarities_pivot)

In [643]:
all_similarities_pivot = all_similarities_pivot.loc[cell_types_exclude_wo_HSPC, (slice(None), cell_types_exclude_wo_HSPC)]

In [ ]:
cytokines_to_plot = [
    "4-1BBL", "APRIL", "IL-2", "IL-15",
    "IL-4", "IL-13", "IL-21", "CD40L",
    "M-CSF", "GM-CSF", "IL-3", "C5a",
    "TSLP", "IL-32-beta", "IFN-omega", "IFN-gamma",
]

# Determine grid dimensions; here we force a 4×4 grid.
n_plots = len(cytokines_to_plot)
n_cols = 4
n_rows = int(np.ceil(n_plots / n_cols))

# Create a figure with subplots.
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=cm_to_inches((18, 16)))
# Flatten the axes array for easy iteration.
axes = axes.flatten()

# Loop over each cytokine and plot its heatmap in the corresponding subplot.
for i, cytokine in enumerate(cytokines_to_plot):
    ax = axes[i]
    
    # Extract the data for this cytokine.
    data = all_similarities_pivot.xs(cytokine, axis=1, level=0)
    
    # Plot the heatmap without its own colorbar.
    sns.heatmap(data,
                cmap="rocket",
                vmin=all_similarities_pivot.min().min(), vmax=all_similarities_pivot.max().max(),
                cbar=False,
                ax=ax,
                square=True)
    
    # Give each subplot its own title.
    ax.set_title(f"{cytokine}", fontsize=7)
    
    ax.set_xlabel("")
    ax.set_ylabel("")
    
    # Remove tick labels from interior subplots:
    # - Only show x tick labels on the bottom row.
    # - Only show y tick labels on the left column.
    row = i // n_cols
    col = i % n_cols
    if row != n_rows - 1:  # not bottom row
        ax.set_xticklabels([])
    if col != 0:          # not left column
        ax.set_yticklabels([])
    
    # Optionally remove tick marks.
    ax.tick_params(length=0)

# Remove any unused (empty) subplots (if the total number of cytokines is less than n_rows*n_cols)
for j in range(n_plots, len(axes)):
    fig.delaxes(axes[j])

# Add one shared colorbar on the right side.
# First, create a ScalarMappable with the same normalization and colormap as used in the heatmaps.
norm = plt.Normalize(vmin=all_similarities_pivot.min().min(), vmax=all_similarities_pivot.max().max())
sm = plt.cm.ScalarMappable(cmap="rocket", norm=norm)
sm.set_array([])

# Adjust the figure layout to make room for the colorbar.
fig.subplots_adjust(right=0.85, wspace=0.3, hspace=0.3)
cbar_ax = fig.add_axes([0.88, 0.15, 0.02, 0.35])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label("Similarity")

# apply tight_layout 
plt.tight_layout()

# Save the complete grid as PNG and SVG.
for fmt in ["png", "svg"]:
    fig.savefig(os.path.join(plot_folder, f"6.2_alternative_similarity_grid.{fmt}"), dpi=300)

plt.show()

In [646]:
# set all diagonals to NaN
for cell_type in all_similarities_pivot.index:
    all_similarities_pivot.loc[cell_type, (slice(None), cell_type)] = np.nan

def get_results_from_eval_subset(df, eval_subset):
    df = df.loc[eval_subset, (slice(None), eval_subset)]
    # name the index cell_type_1 and the second column level cell_type_2
    # rename the index and column levels
    df.index.name = "cell_type_1"
    df.columns.names = ["cytokine", "cell_type_2"]

    # flatten mean_cell_cell_alt_sim
    df = df.melt(ignore_index=False).dropna()
    mean_cell_cell_alt_sim = df.groupby("cytokine")["value"].mean()
    std_cell_cell_alt_sim = df.groupby("cytokine")["value"].std()
    
    return mean_cell_cell_alt_sim, std_cell_cell_alt_sim
    
cell_eval_subset = ["B cell", "CD4 T cell", "NK CD56low", "cDC", "CD14 Mono"]
mean_cell_cell_alt_sim, std_cell_cell_alt_sim =  get_results_from_eval_subset(all_similarities_pivot.copy(), cell_eval_subset)

other_subset = [
    ["B cell", "CD4 T cell"],
    ["B cell", "NK CD56low"],
    ["B cell", "cDC"],
    ["B cell", "CD14 Mono"],
]
mean_by_subset = {}
for subset in other_subset:
    mean_by_subset[f"{subset[0]}-{subset[1]}"], _ =  get_results_from_eval_subset(all_similarities_pivot.copy(), subset)

In [ ]:
# create a line plot of mean_cell_cell_alt_sim with the cytokine names as the x-axis, ordered from low to high mean pearson r
plt.figure(figsize=cm_to_inches((18,2.7)))
sorted_vals = mean_cell_cell_alt_sim.sort_values()
plt.plot(range(len(sorted_vals)), sorted_vals, '-', markersize=2, color="black", label="mean")
plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-cDC"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["cDC"], alpha=0.5, label="B cell-cDC")
plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-CD14 Mono"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["CD14 Mono"], alpha=0.5, label="B cell-CD14 Mono")
plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-CD4 T cell"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["CD4 T cell"], alpha=0.5, label="B cell-CD4 T cell")
plt.plot(range(len(sorted_vals)), mean_by_subset["B cell-NK CD56low"][sorted_vals.index], '-', markersize=2, color=CELL_TYPE_COLORS["NK CD56low"], alpha=0.5, label="B cell-NK CD56low")
plt.legend(ncol=5)

# Find the index where mean goes above 0.6
threshold = 0.6
above_threshold_indices = np.where(sorted_vals.values > threshold)[0]
if len(above_threshold_indices) > 0:
    first_above_threshold_idx = above_threshold_indices[0]
    # Add a vertical dashed line at the threshold point
    plt.axvline(x=first_above_threshold_idx, color='red', linestyle='--', linewidth=1)
    # Add text annotation
    plt.text(first_above_threshold_idx + 1, 0.3, f'r < {threshold}', 
             color='red', fontsize=7, verticalalignment='bottom')

# Plot standard deviation as shaded area
plt.fill_between(range(len(sorted_vals)), 
                 sorted_vals - std_cell_cell_alt_sim[sorted_vals.index],
                 sorted_vals + std_cell_cell_alt_sim[sorted_vals.index],
                 alpha=0.2, color='gray')
plt.xticks(range(len(sorted_vals)), sorted_vals.index, rotation=90, ha='center', fontsize=5.5)
plt.ylabel('mean cell-cell\nsimilarity')
# plt.ylim(0.25, 0.9)
apply_greek_labels_heatmap(plt.gca())
#plt.grid(True, alpha=0.3)
plt.margins(x=0.01) # Remove whitespace on left and right
plt.tight_layout()
plt.savefig(os.path.join(plot_folder, "6.2_mean_alt_sim_vals_ordered.svg")) 